
<div style="background: linear-gradient(135deg, #1A1040 0%, #2D1B69 100%); padding: 40px 48px; border-radius: 12px; margin-bottom: 8px;">
  <div style="color: #F5A623; font-size: 13px; font-weight: 700; letter-spacing: 3px; text-transform: uppercase; margin-bottom: 8px;">BITS Pilani â€” Professional AI/ML Programme</div>
  <div style="color: white; font-size: 36px; font-weight: 900; margin-bottom: 6px;">AWS Model Engineering Lab</div>
  <div style="color: #C4B5FD; font-size: 18px; margin-bottom: 20px;">Lab 1: Build & Register &nbsp;Â·&nbsp; Lab 1.1: Deploy & Operationalize</div>
  <div style="display: flex; gap: 16px; flex-wrap: wrap; margin-top: 16px;">
    <span style="background: rgba(245,166,35,0.2); border: 1px solid #F5A623; color: #F5A623; padding: 6px 14px; border-radius: 20px; font-size: 13px; font-weight: 600;">â± 2.5 Hours</span>
    <span style="background: rgba(255,255,255,0.1); color: white; padding: 6px 14px; border-radius: 20px; font-size: 13px; font-weight: 600;">2 Labs Â· 9 Steps</span>
    <span style="background: rgba(255,255,255,0.1); color: white; padding: 6px 14px; border-radius: 20px; font-size: 13px; font-weight: 600;">S3 Â· SageMaker Â· EventBridge</span>
  </div>
</div>

### ðŸ“‹ What This Notebook Covers

| Phase | Time | Content |
|-------|------|---------|
| **Setup** | 0â€“10 min | Environment config, S3 paths, dataset preparation |
| **Lab 1 â€” Steps 1â€“3** | 10â€“30 min | Freeze dataset, split contract, EDA Processing job |
| **Lab 1 â€” Steps 4â€“5** | 30â€“55 min | Feature pipeline, feature selection & reduction |
| **Lab 1 â€” Steps 6â€“7** | 55â€“80 min | Model bakeoff (3 candidates), standardised evaluation |
| **Lab 1 â€” Steps 8â€“9** | 80â€“95 min | Registry registration, model card |
| **Break** | 95â€“105 min | â˜• Q&A buffer |
| **Lab 1.1 â€” Steps 1â€“3** | 105â€“115 min | Pull from registry, production validation, approve |
| **Lab 1.1 â€” Steps 4â€“5** | 115â€“125 min | Package assets, deploy endpoint |
| **Lab 1.1 â€” Steps 6â€“8** | 125â€“138 min | EventBridge schedule, event hooks, clustering sidecar |
| **Wrap-up** | 138â€“150 min | Handoff checklists, deliverables sign-off |

> **Dataset used throughout:** UCI Bank Marketing dataset (binary classification â€” term deposit subscription)  
> **Target column:** `y` (yes/no â†’ 1/0)



---
## ðŸ”§ Section 0 â€” Environment Setup & Configuration
*Run all cells in this section before starting Lab 1.*


In [1]:
!apt-get update -qq
!apt-get install -y python3.10 python3.10-venv python3.10-dev

!python3.10 -m venv /content/lab_env

!/content/lab_env/bin/pip install --upgrade pip setuptools wheel
!/content/lab_env/bin/pip install \
    "numpy==1.26.4" \
    "scikit-learn==1.2.2" \
    "joblib==1.3.2" \
    "pandas==2.0.3" \
    "boto3" \
    "sagemaker"

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpython3.10 libpython3.10-dev libpython3.10-minimal libpython3.10-stdlib
  python3.10-distutils python3.10-lib2to3 python3.10-minimal
Suggested packages:
  binfmt-support
The following NEW packages will be installed:
  libpython3.10 libpython3.10-dev libpython3.10-minimal libpython3.10-stdlib
  python3.10 python3.10-dev python3.10-distutils python3.10-lib2to3
  python3.10-minimal python3.10-venv
0 upgraded, 10 newly installed, 0 to remove an

In [2]:
%%writefile /content/train_model.py
import joblib
import numpy as np
import sklearn
from sklearn.ensemble import RandomForestClassifier

print(f"NumPy Version: {np.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")

# Example Training Logic
X = np.array([[1, 2], [3, 4], [5, 6]])
y = np.array([0, 1, 0])

clf = RandomForestClassifier(n_estimators=10, random_state=42)
clf.fit(X, y)

# Save artifact binary-compatible with SageMaker Scikit-Learn 1.2-1 container
joblib.dump(clf, "model.joblib")
print("Successfully saved model.joblib in Python 3.10 environment!")

Writing /content/train_model.py


In [3]:
!/content/lab_env/bin/python /content/train_model.py

NumPy Version: 1.26.4
Scikit-Learn Version: 1.2.2
Successfully saved model.joblib in Python 3.10 environment!


In [4]:
import os
import subprocess
import sys

# Path to Python 3.10 virtual environment executable
venv_python = "/content/lab_env/bin/python"


def run_venv(code: str):
    """Executes Python code directly inside the Python 3.10 virtual environment."""
    # Override Colab's notebook backend with the headless 'Agg' backend for subprocess execution
    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"

    res = subprocess.run(
        [venv_python, "-c", code], capture_output=True, text=True, env=env
    )
    if res.stdout:
        print(res.stdout, end="")
    if res.stderr:
        print(res.stderr, file=sys.stderr, end="")

In [5]:
run_venv(
    r"""
import sys
import joblib
import numpy as np
import sklearn

print(f"NumPy Version: {np.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")
print(f"Joblib Version: {joblib.__version__}")

# Assert exact required versions for AWS SageMaker compatibility
assert np.__version__ == "1.26.4", f"Expected NumPy 1.26.4, got {np.__version__}"
assert sklearn.__version__ == "1.2.2", f"Expected Scikit-Learn 1.2.2, got {sklearn.__version__}"

print("\nEnvironment verification passed successfully!")
"""
)

NumPy Version: 1.26.4
Scikit-Learn Version: 1.2.2
Joblib Version: 1.3.2

Environment verification passed successfully!


In [6]:
import os
from pathlib import Path
try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_colab_secret(name, required=True):
    try:
        value = os.environ.get(name) or userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f"Unable to read Colab Secret: {name}") from exc
        return None
    if required and not value:
        raise RuntimeError(
            f"Add the Colab Secret {name} and grant this notebook access."
        )
    return value


# Clone repository if missing
REPO_ROOT = Path("/content/BITS_programming")
if not REPO_ROOT.exists():
    !git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming

# Set working directory for host and future run_venv processes
NOTEBOOK_DIR = REPO_ROOT / "assignments/assignment_9"
os.chdir(NOTEBOOK_DIR)
print(f"Working directory updated to: {os.getcwd()}")

Cloning into '/content/BITS_programming'...
remote: Enumerating objects: 2315, done.
remote: Counting objects: 100% (249/249), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 2315 (delta 112), reused 180 (delta 77), pack-reused 2066 (from 1)
Receiving objects: 100% (2315/2315), 269.62 MiB | 19.60 MiB/s, done.
Resolving deltas: 100% (501/501), done.
Updating files: 100% (1317/1317), done.
Working directory updated to: /content/BITS_programming/assignments/assignment_9


In [7]:
# 1. Inject AWS Secrets into Host Environment (inherited by run_venv sub-processes)
import os

os.environ["AWS_ACCESS_KEY_ID"] = get_colab_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = get_colab_secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

# 2. Execute AWS setup and version checks inside Python 3.10 virtual environment
run_venv(r"""
import os, json, time, hashlib, datetime, warnings
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, yaml
import sklearn

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", palette="muted")

# boto3 setup inside virtual environment
REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
boto_session = boto3.session.Session(region_name=REGION)

sm_client  = boto_session.client("sagemaker")
s3_client  = boto_session.client("s3")
eb_client  = boto_session.client("events")
iam_client = boto_session.client("iam")
sts_client = boto_session.client("sts")

caller = sts_client.get_caller_identity()
ACCOUNT_ID = caller["Account"]
CALLER_ARN = caller["Arn"]

def resolve_sagemaker_execution_role(caller_arn: str) -> str:
    if ":assumed-role/" in caller_arn:
        role_name = caller_arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
        try:
            return iam_client.get_role(RoleName=role_name)["Role"]["Arn"]
        except Exception:
            return (
                caller_arn.replace("arn:aws:sts::", "arn:aws:iam::")
                          .replace(":assumed-role/", ":role/")
                          .rsplit("/", 1)[0]
            )
    return caller_arn

ROLE = resolve_sagemaker_execution_role(CALLER_ARN)

print(f"AWS caller : {CALLER_ARN}")
print(f"Role       : {ROLE}")
print(f"Region     : {REGION}")
print(f"Account ID : {ACCOUNT_ID}")

# Packaging compatibility check
print(f"NumPy      : {np.__version__}")
print(f"sklearn    : {sklearn.__version__}")
print(f"joblib     : {joblib.__version__}")

if int(np.__version__.split('.')[0]) >= 2:
    print("⚠️ NumPy 2.x detected. Restart kernel after running the install cell.")
else:
    print("✅ Environment successfully verified on NumPy 1.x / Scikit-Learn 1.2.2 matrix.")
"""
)

Traceback (most recent call last):
  File "<string>", line 7, in <module>
ModuleNotFoundError: No module named 'seaborn'


In [8]:
run_venv(
    r"""
import os
import boto3

# ── S3 bucket & path configuration ──────────────────────────────────────────
PROJECT = "model-engineering-lab"
DATASET_VERSION = "v1"
SPLIT_VERSION = "split_v1"
PREPROCESS_VERSION = "preprocess_v1"
DEPLOYMENT_INSTANCE_TYPE = "ml.m5.large"

# Re-initialize boto3 clients in subprocess scope
REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
boto_session = boto3.session.Session(region_name=REGION)
s3_client = boto_session.client("s3")
sts_client = boto_session.client("sts")

ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

# SageMaker-style default bucket name, created with boto3 if missing.
BUCKET = f"sagemaker-{REGION}-{ACCOUNT_ID}"

try:
    s3_client.head_bucket(Bucket=BUCKET)
    print(f"ℹ️ Bucket already exists: {BUCKET}")
except Exception:
    if REGION == "us-east-1":
        s3_client.create_bucket(Bucket=BUCKET)
    else:
        s3_client.create_bucket(
            Bucket=BUCKET,
            CreateBucketConfiguration={"LocationConstraint": REGION},
        )
    print(f"✅ Created bucket: {BUCKET}")

# Root S3 prefix
PREFIX = f"{PROJECT}"

# Derived S3 paths — do not edit
PATHS = {
    "raw":              f"s3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/raw",
    "schema":           f"s3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/schema",
    "manifest":         f"s3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/manifest.json",
    "splits":           f"s3://{BUCKET}/{PREFIX}/splits/{SPLIT_VERSION}",
    "eda":              f"s3://{BUCKET}/{PREFIX}/eda",
    "pipelines":        f"s3://{BUCKET}/{PREFIX}/pipelines",
    "feature_configs":  f"s3://{BUCKET}/{PREFIX}/feature_configs",
    "features":         f"s3://{BUCKET}/{PREFIX}/features",
    "runs":             f"s3://{BUCKET}/{PREFIX}/runs",
    "evaluation":       f"s3://{BUCKET}/{PREFIX}/evaluation",
    "registry":         f"s3://{BUCKET}/{PREFIX}/registry",
    "model_cards":      f"s3://{BUCKET}/{PREFIX}/model_cards",
    "release":          f"s3://{BUCKET}/{PREFIX}/release",
    "deploy":           f"s3://{BUCKET}/{PREFIX}/deploy",
    "unsupervised":     f"s3://{BUCKET}/{PREFIX}/unsupervised",
}

print("✅ S3 paths configured")
for k, v in PATHS.items():
    print(f"  {k:<20} -> {v}")
"""
)

ℹ️ Bucket already exists: sagemaker-us-east-1-455865672536
✅ S3 paths configured
  raw                  -> s3://sagemaker-us-east-1-455865672536/model-engineering-lab/datasets/v1/raw
  schema               -> s3://sagemaker-us-east-1-455865672536/model-engineering-lab/datasets/v1/schema
  manifest             -> s3://sagemaker-us-east-1-455865672536/model-engineering-lab/datasets/v1/manifest.json
  splits               -> s3://sagemaker-us-east-1-455865672536/model-engineering-lab/splits/split_v1
  eda                  -> s3://sagemaker-us-east-1-455865672536/model-engineering-lab/eda
  pipelines            -> s3://sagemaker-us-east-1-455865672536/model-engineering-lab/pipelines
  feature_configs      -> s3://sagemaker-us-east-1-455865672536/model-engineering-lab/feature_configs
  features             -> s3://sagemaker-us-east-1-455865672536/model-engineering-lab/features
  runs                 -> s3://sagemaker-us-east-1-455865672536/model-engineering-lab/runs
  evaluation           -

In [9]:
%%writefile /content/lab_helpers.py
import os, json, io, datetime
import boto3
import pandas as pd

def _get_s3_client():
    REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
    return boto3.session.Session(region_name=REGION).client("s3")

def s3_upload_json(obj, s3_path):
    """Upload a dict as JSON to an s3:// path."""
    bucket, key = s3_path.replace("s3://", "").split("/", 1)
    body = json.dumps(obj, indent=2, default=str)
    _get_s3_client().put_object(Bucket=bucket, Key=key, Body=body.encode())
    print(f"   ✅ uploaded → {s3_path}")

def s3_upload_bytes(data: bytes, s3_path: str, content_type="application/octet-stream"):
    bucket, key = s3_path.replace("s3://", "").split("/", 1)
    _get_s3_client().put_object(Bucket=bucket, Key=key, Body=data, ContentType=content_type)
    print(f"   ✅ uploaded → {s3_path}")

def s3_upload_text(text: str, s3_path: str):
    s3_upload_bytes(text.encode(), s3_path, "text/plain")

def s3_download_json(s3_path):
    bucket, key = s3_path.replace("s3://", "").split("/", 1)
    obj = _get_s3_client().get_object(Bucket=bucket, Key=key)
    return json.loads(obj["Body"].read())

def s3_download_df(s3_path):
    return pd.read_parquet(s3_path)

def s3_upload_df(df, s3_path):
    buf = io.BytesIO()
    df.to_parquet(buf, index=True)
    s3_upload_bytes(buf.getvalue(), s3_path, "application/octet-stream")

def now_iso():
    return datetime.datetime.now(datetime.timezone.utc).isoformat().replace("+00:00", "Z")

Writing /content/lab_helpers.py


In [10]:
run_venv(
    r"""
import sys
sys.path.insert(0, "/content")

import lab_helpers

print("✅ Helper utilities loaded and verified inside lab_env!")
"""
)

✅ Helper utilities loaded and verified inside lab_env!



---
<div style="background: #1D4ED8; padding: 24px 32px; border-radius: 10px; color: white; margin: 16px 0;">
  <div style="font-size: 22px; font-weight: 900; margin-bottom: 4px;">ðŸ§ª LAB 1 â€” BUILD & REGISTER A CANDIDATE</div>
  <div style="font-size: 14px; opacity: 0.85;">Steps 1â€“9 &nbsp;Â·&nbsp; ~85 minutes &nbsp;Â·&nbsp; S3 Â· SageMaker Processing Â· Training Â· Model Registry</div>
</div>

**Objective:** Create a repeatable offline training pipeline that is leakage-safe, tests three feature configurations, compares 2â€“3 supervised models, and registers the best candidate in SageMaker Model Registry with a full model card.



---
### ðŸ“¦ Step 1 â€” Freeze the Dataset
**â± ~10 min &nbsp;|&nbsp; Service: S3**

Store one immutable snapshot. This is the traceability anchor for all downstream registry and lineage workflows.  
**Rule:** No preprocessing, no sampling â€” raw data exactly as received.

> ðŸ’¡ **Teaching Point:** A dataset that changes between runs breaks reproducibility and makes model comparisons meaningless. Freeze first, fit later.


In [11]:
run_venv(
    r"""
import zipfile
from urllib.request import urlretrieve
import pandas as pd

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip"
urlretrieve(URL, "/tmp/bank-additional.zip")

with zipfile.ZipFile("/tmp/bank-additional.zip") as z:
    z.extractall("/tmp/bank/")

raw_df = pd.read_csv(
    "/tmp/bank/bank-additional/bank-additional-full.csv",
    sep=";", na_values=["unknown"]
)

print(f"Shape  : {raw_df.shape}")
print(f"Target : {raw_df['y'].value_counts().to_dict()}\n")
print(raw_df.head(3))
"""
)

Shape  : (41188, 21)
Target : {'no': 36548, 'yes': 4640}

   age        job  marital  ... euribor3m nr.employed   y
0   56  housemaid  married  ...     4.857      5191.0  no
1   57   services  married  ...     4.857      5191.0  no
2   37   services  married  ...     4.857      5191.0  no

[3 rows x 21 columns]


In [12]:
run_venv(
    r"""
import sys
sys.path.insert(0, "/content")

import os, json, hashlib, io
import pandas as pd
import boto3
from lab_helpers import s3_upload_bytes, s3_upload_json, now_iso

# ── Configuration & Environment Setup ───────────────────────────────────────
PROJECT = "model-engineering-lab"
DATASET_VERSION = "v1"
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip"

REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
boto_session = boto3.session.Session(region_name=REGION)
sts_client = boto_session.client("sts")
ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

BUCKET = f"sagemaker-{REGION}-{ACCOUNT_ID}"
PREFIX = f"{PROJECT}"

PATHS = {
    "raw":      f"s3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/raw",
    "schema":   f"s3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/schema",
    "manifest": f"s3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/manifest.json",
}

# Re-load dataframe from local unzipped path
raw_df = pd.read_csv(
    "/tmp/bank/bank-additional/bank-additional-full.csv",
    sep=";", na_values=["unknown"]
)

# ── 1. Upload raw snapshot to S3 ─────────────────────────────────────────────
raw_s3 = f"{PATHS['raw']}/bank_marketing_raw.parquet"
buf = io.BytesIO()
raw_df.to_parquet(buf, index=False)
s3_upload_bytes(buf.getvalue(), raw_s3)

# Compute content hash for immutability proof
raw_hash = hashlib.md5(buf.getvalue()).hexdigest()
print(f"   Content MD5 : {raw_hash}")

# ── 2. Write schema.json ──────────────────────────────────────────────────────
schema = {
    "columns": {
        col: str(dtype)
        for col, dtype in raw_df.dtypes.items()
    },
    "target_column": "y",
    "target_values": ["yes", "no"],
    "n_columns": int(raw_df.shape[1]),
    "nullable_columns": raw_df.columns[raw_df.isnull().any()].tolist()
}

schema_s3 = f"{PATHS['schema']}/schema.json"
s3_upload_json(schema, schema_s3)

# ── 3. Write manifest.json ───────────────────────────────────────────────────
manifest = {
    "dataset_version":    DATASET_VERSION,
    "source":             URL,
    "raw_s3_path":        raw_s3,
    "schema_s3_path":     schema_s3,
    "row_count":          int(len(raw_df)),
    "column_count":       int(raw_df.shape[1]),
    "content_md5":        raw_hash,
    "created_at":         now_iso(),
    "data_quality": {
        "missing_cells":  int(raw_df.isnull().sum().sum()),
        "missing_pct":    round(raw_df.isnull().mean().mean() * 100, 2),
        "duplicate_rows": int(raw_df.duplicated().sum()),
        "class_balance":  raw_df["y"].value_counts(normalize=True).round(4).to_dict(),
    }
}

s3_upload_json(manifest, PATHS["manifest"])
print("\n📋 Manifest summary:")
print(json.dumps(manifest["data_quality"], indent=2))

# ── 4. Verify freeze ─────────────────────────────────────────────────────────
restored = pd.read_parquet(raw_s3)
assert restored.shape == raw_df.shape, "Shape mismatch after round-trip!"
print(f"\n✅ Dataset frozen and verified: {restored.shape[0]:,} rows × {restored.shape[1]} cols")
print(f"   Dataset version: {DATASET_VERSION}")
print(f"   MD5:             {raw_hash}")
"""
)

   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/datasets/v1/raw/bank_marketing_raw.parquet
   Content MD5 : 19814c0ebedb55cc763b6f11173416f4
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/datasets/v1/schema/schema.json
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/datasets/v1/manifest.json

📋 Manifest summary:
{
  "missing_cells": 12718,
  "missing_pct": 1.47,
  "duplicate_rows": 12,
  "class_balance": {
    "no": 0.8873,
    "yes": 0.1127
  }
}

✅ Dataset frozen and verified: 41,188 rows × 21 cols
   Dataset version: v1
   MD5:             19814c0ebedb55cc763b6f11173416f4


/content/lab_env/lib/python3.10/site-packages/fsspec/registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)



---
### âœ‚ï¸ Step 2 â€” Create Split Definitions (Evaluation Contract)
**â± ~5 min &nbsp;|&nbsp; Service: S3**

Generate and persist split indices **before** fitting anything. These indices become your production evaluation contract â€” every model candidate must use exactly these splits.

> ðŸ’¡ **Teaching Point:** Any model that saw a different validation set cannot be fairly compared. The split contract enforces this guarantee.


In [13]:
run_venv(
    r"""
import sys
sys.path.insert(0, "/content")

import os, io, json
import pandas as pd
import boto3
from sklearn.model_selection import StratifiedKFold, train_test_split
from lab_helpers import s3_upload_bytes, s3_upload_json, s3_upload_df, now_iso

# ── Configuration & S3 Path Setup ───────────────────────────────────────────
PROJECT = "model-engineering-lab"
DATASET_VERSION = "v1"
SPLIT_VERSION = "split_v1"

REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
boto_session = boto3.session.Session(region_name=REGION)
sts_client = boto_session.client("sts")
ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

BUCKET = f"sagemaker-{REGION}-{ACCOUNT_ID}"
PREFIX = f"{PROJECT}"

PATHS = {
    "raw":    f"s3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/raw",
    "splits": f"s3://{BUCKET}/{PREFIX}/splits/{SPLIT_VERSION}",
}

# ── 1. Encode target & build stratified train/val/test split ─────────────────
raw_df = pd.read_csv(
    "/tmp/bank/bank-additional/bank-additional-full.csv",
    sep=";", na_values=["unknown"]
)

df = raw_df.copy()
df["target"] = (df["y"] == "yes").astype(int)
df = df.drop(columns=["y"])

# Remove exact duplicates after encoding
df = df.drop_duplicates().reset_index(drop=True)

all_idx = df.index.tolist()

# 60% train / 20% val / 20% test — stratified
train_idx, temp_idx = train_test_split(
    all_idx, test_size=0.40, random_state=42,
    stratify=df.loc[all_idx, "target"]
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=42,
    stratify=df.loc[temp_idx, "target"]
)

print(f"Train : {len(train_idx):>6,}  |  pos rate: {df.loc[train_idx,'target'].mean():.3f}")
print(f"Val   : {len(val_idx):>6,}  |  pos rate: {df.loc[val_idx,'target'].mean():.3f}")
print(f"Test  : {len(test_idx):>6,}  |  pos rate: {df.loc[test_idx,'target'].mean():.3f}")

# ── 2. 5-fold CV definitions (for feature selection stability) ───────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_train_raw = df.loc[train_idx]
y_train_raw = df.loc[train_idx, "target"]

cv_folds = []
for fold_i, (fi_train, fi_val) in enumerate(skf.split(X_train_raw, y_train_raw)):
    cv_folds.append({
        "fold": fold_i,
        "train_indices": X_train_raw.iloc[fi_train].index.tolist(),
        "val_indices":   X_train_raw.iloc[fi_val].index.tolist(),
    })

print(f"\n✅ {len(cv_folds)} CV folds generated")
print(f"   Fold 0 train: {len(cv_folds[0]['train_indices']):,}  |  val: {len(cv_folds[0]['val_indices']):,}")

# ── 3. Persist split indices to S3 ───────────────────────────────────────────
def idx_to_parquet_bytes(indices):
    buf = io.BytesIO()
    pd.DataFrame({"idx": indices}).to_parquet(buf, index=False)
    return buf.getvalue()

s3_upload_bytes(idx_to_parquet_bytes(train_idx), f"{PATHS['splits']}/train_idx.parquet")
s3_upload_bytes(idx_to_parquet_bytes(val_idx),   f"{PATHS['splits']}/valid_idx.parquet")
s3_upload_bytes(idx_to_parquet_bytes(test_idx),  f"{PATHS['splits']}/test_idx.parquet")

s3_upload_json({
    "folds": cv_folds,
    "n_folds": 5,
    "random_state": 42,
    "split_version": SPLIT_VERSION,
    "created_at": now_iso()
}, f"{PATHS['splits']}/cv_folds.json")

# Save the processed dataframe (used by all subsequent steps)
s3_upload_df(df, f"{PATHS['raw']}/bank_marketing_processed.parquet")

print("\n✅ Split contract persisted. Split version:", SPLIT_VERSION)

# ── 4. Split Metadata ────────────────────────────────────────────────────────
split_meta = {
    "split_type":   "stratified_random",
    "time_column":  None,
    "group_column": None,
    "train_cutoff": None,
    "val_cutoff":   None,
}
s3_upload_json(split_meta, f"{PATHS['splits']}/split_meta.json")
print("✅ Split metadata persisted (update split_type if using temporal or group splits)")
"""
)

Train : 24,705  |  pos rate: 0.113
Val   :  8,235  |  pos rate: 0.113
Test  :  8,236  |  pos rate: 0.113

✅ 5 CV folds generated
   Fold 0 train: 19,764  |  val: 4,941
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/splits/split_v1/train_idx.parquet
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/splits/split_v1/valid_idx.parquet
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/splits/split_v1/test_idx.parquet
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/splits/split_v1/cv_folds.json
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/datasets/v1/raw/bank_marketing_processed.parquet

✅ Split contract persisted. Split version: split_v1
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/splits/split_v1/split_meta.json
✅ Split metadata persisted (update split_type if using temporal or group splits)



---
### ðŸ” Step 3 â€” EDA & Leakage Scan
**â± ~10 min &nbsp;|&nbsp; Service: S3 (inline â€” would be a Processing job in production)**

Run preprocessing diagnostics **on training data only**. In production this runs as a SageMaker Processing job to make the EDA reproducible and auditable.


In [14]:
run_venv( r"""
import sys
sys.path.insert(0, "/content")

import os, io, json
import pandas as pd
import numpy as np
import boto3
import matplotlib.pyplot as plt
from lab_helpers import s3_upload_bytes, s3_upload_json, s3_upload_text, now_iso

# ── Configuration & S3 Path Setup ───────────────────────────────────────────
PROJECT = "model-engineering-lab"
DATASET_VERSION = "v1"
SPLIT_VERSION = "split_v1"

REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
boto_session = boto3.session.Session(region_name=REGION)
sts_client = boto_session.client("sts")
ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

BUCKET = f"sagemaker-{REGION}-{ACCOUNT_ID}"
PREFIX = f"{PROJECT}"

PATHS = {
    "raw":    f"s3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/raw",
    "splits": f"s3://{BUCKET}/{PREFIX}/splits/{SPLIT_VERSION}",
    "eda":    f"s3://{BUCKET}/{PREFIX}/eda",
}

# ── 1. Load training split ───────────────────────────────────────────────────
df = pd.read_parquet(f"{PATHS['raw']}/bank_marketing_processed.parquet")
train_idx = pd.read_parquet(f"{PATHS['splits']}/train_idx.parquet")["idx"].tolist()

train_df = df.loc[train_idx].copy()
X_tr = train_df.drop(columns=["target"])
y_tr = train_df["target"]

num_cols = X_tr.select_dtypes(include="number").columns.tolist()
cat_cols = X_tr.select_dtypes(exclude="number").columns.tolist()

print(f"Numeric   columns ({len(num_cols)}): {num_cols}")
print(f"Categoric columns ({len(cat_cols)}): {cat_cols}")

# ── 2. Missing value analysis ────────────────────────────────────────────────
miss_pct = (X_tr.isnull().sum() / len(X_tr) * 100).sort_values(ascending=False)
high_miss = miss_pct[miss_pct > 30]

print("\nHigh-missingness columns (>30%):")
print(high_miss if len(high_miss) > 0 else "  None")

fig, ax = plt.subplots(figsize=(10, 3))
miss_pct[miss_pct > 0].plot(kind="barh", ax=ax, color="#2D1B69")
ax.set_title("Missing Value % (training set)", fontweight="bold")
ax.set_xlabel("% Missing")
plt.tight_layout()
plt.savefig("/tmp/missing_values.png")
plt.close()

# ── 3. ID-like & near-constant feature detection ─────────────────────────────
id_like = [c for c in X_tr.columns if X_tr[c].nunique() / len(X_tr) > 0.95]
near_const = [c for c in num_cols if X_tr[c].std() < 1e-5]
variance_threshold_candidates = [c for c in num_cols if X_tr[c].var() < 0.01]

print("\nID-like columns (>95% unique):", id_like or "None")
print("Near-constant columns:", near_const or "None")
print("Low-variance candidates (<0.01):", variance_threshold_candidates or "None")

# ── 4. Leakage scan — correlation with target ────────────────────────────────
train_enc = pd.get_dummies(X_tr.fillna("MISSING"), drop_first=True)
corr_with_target = train_enc.corrwith(y_tr).abs().sort_values(ascending=False)

leakage_threshold = 0.85
leakage_candidates = corr_with_target[corr_with_target > leakage_threshold]

print(f"\nTop-10 correlations with target:")
print(corr_with_target.head(10).to_string())

if len(leakage_candidates) > 0:
    print(f"\n⚠️  LEAKAGE RISK — {len(leakage_candidates)} features above {leakage_threshold}:")
    print(leakage_candidates.to_string())
else:
    print(f"\n✅ No features above leakage threshold ({leakage_threshold})")

# ── 5. Cardinality check for categoricals ────────────────────────────────────
cardinality = X_tr[cat_cols].nunique().sort_values(ascending=False)
high_cardinality = cardinality[cardinality > 20].index.tolist()

print("\nCardinality of categorical columns:")
print(cardinality.to_string())
print(f"\nHigh-cardinality (>20): {high_cardinality}")

# ── 6. Skew analysis ─────────────────────────────────────────────────────────
skew = X_tr[num_cols].skew().sort_values(key=abs, ascending=False)
high_skew = skew[skew.abs() > 1.5]

print("\nHigh-skew features (|skew| > 1.5):")
print(high_skew.to_string() if len(high_skew) > 0 else "  None")

# ── 7. Persist EDA artifacts to S3 ───────────────────────────────────────────
feature_profile = pd.DataFrame({
    "dtype":       X_tr.dtypes.astype(str),
    "n_unique":    X_tr.nunique(),
    "miss_pct":    miss_pct,
    "skew":        X_tr[num_cols].skew().reindex(X_tr.columns),
    "corr_target": corr_with_target.reindex(X_tr.columns),
}).reset_index().rename(columns={"index": "feature"})

buf = io.BytesIO()
feature_profile.to_parquet(buf, index=False)
s3_upload_bytes(buf.getvalue(), f"{PATHS['eda']}/feature_profile.parquet")

leakage_risks = {
    "threshold": leakage_threshold,
    "flagged_features": leakage_candidates.index.tolist(),
    "correlations": leakage_candidates.round(4).to_dict(),
    "id_like_features": id_like,
    "near_constant_features": near_const,
    "created_at": now_iso()
}
s3_upload_json(leakage_risks, f"{PATHS['eda']}/leakage_risks.json")

s3_upload_json({
    "high_cardinality_columns": high_cardinality,
    "cardinality": cardinality.to_dict()
}, f"{PATHS['eda']}/high_cardinality_columns.json")

transform_candidates = {
    "log_transform": high_skew.index.tolist(),
    "high_missingness_drop": high_miss.index.tolist(),
    "low_variance_drop": variance_threshold_candidates,
}
s3_upload_json(transform_candidates, f"{PATHS['eda']}/transformation_candidates.json")

eda_summary = f'''# EDA Summary — {DATASET_VERSION}
Generated: {now_iso()}

## Dataset
- Rows (train): {len(train_df):,}
- Numeric features: {len(num_cols)}
- Categorical features: {len(cat_cols)}
- Target positive rate: {y_tr.mean():.3f}

## Leakage Scan
- Threshold: {leakage_threshold}
- Flagged: {leakage_candidates.index.tolist() or 'None'}
- ID-like columns: {id_like or 'None'}

## Missing Values
- High missingness (>30%): {high_miss.index.tolist() or 'None'}

## Cardinality
- High cardinality (>20 unique): {high_cardinality or 'None'}

## Skew
- High-skew features: {high_skew.index.tolist() or 'None'}
'''
s3_upload_text(eda_summary, f"{PATHS['eda']}/eda_summary.md")

print("\n✅ All EDA artifacts uploaded to S3")
"""
)

Numeric   columns (10): ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Categoric columns (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']

High-missingness columns (>30%):
  None

ID-like columns (>95% unique): None
Near-constant columns: None
Low-variance candidates (<0.01): None

Top-10 correlations with target:
duration                0.412059
nr.employed             0.347342
pdays                   0.328965
poutcome_success        0.320269
euribor3m               0.300253
emp.var.rate            0.290132
previous                0.232312
poutcome_nonexistent    0.193385
month_mar               0.148096
contact_telephone       0.144976

✅ No features above leakage threshold (0.85)

Cardinality of categorical columns:
job            11
month          10
education       7
day_of_week     5
marital         3
poutcome        3
loan        

/content/lab_env/lib/python3.10/site-packages/fsspec/registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)



---
### âš™ï¸ Step 4 â€” Build the Feature Pipeline
**â± ~15 min &nbsp;|&nbsp; Service: S3**

Implement **train-only** preprocessing: fit on train split only, then transform train/val/test.  
Persist the fitted pipeline artifact so inference uses the exact same transforms.


In [15]:
run_venv(
    r"""
import sys
sys.path.insert(0, "/content")

import os
import io
import json
import yaml
import joblib
import numpy as np
import pandas as pd
import boto3

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, TransformerMixin

from lab_helpers import s3_upload_bytes, s3_upload_json, s3_upload_text, now_iso

# ── Configuration & S3 Path Setup ───────────────────────────────────────────
PROJECT = "model-engineering-lab"
DATASET_VERSION = "v1"
SPLIT_VERSION = "split_v1"
PREPROCESS_VERSION = "preprocess_v1"

REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
boto_session = boto3.session.Session(region_name=REGION)
sts_client = boto_session.client("sts")
ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

BUCKET = f"sagemaker-{REGION}-{ACCOUNT_ID}"
PREFIX = f"{PROJECT}"

PATHS = {
    "raw":             f"s3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/raw",
    "splits":          f"s3://{BUCKET}/{PREFIX}/splits/{SPLIT_VERSION}",
    "pipelines":       f"s3://{BUCKET}/{PREFIX}/pipelines",
    "feature_configs": f"s3://{BUCKET}/{PREFIX}/feature_configs",
}

# ── 1. Reload splits from S3 ────────────────────────────────────────────────
train_idx_loaded = pd.read_parquet(f"{PATHS['splits']}/train_idx.parquet")["idx"].tolist()
val_idx_loaded   = pd.read_parquet(f"{PATHS['splits']}/valid_idx.parquet")["idx"].tolist()
test_idx_loaded  = pd.read_parquet(f"{PATHS['splits']}/test_idx.parquet")["idx"].tolist()

df_full = pd.read_parquet(f"{PATHS['raw']}/bank_marketing_processed.parquet")

X = df_full.drop(columns=["target"])
y = df_full["target"]

X_train = X.loc[train_idx_loaded]
X_val   = X.loc[val_idx_loaded]
X_test  = X.loc[test_idx_loaded]
y_train = y.loc[train_idx_loaded]
y_val   = y.loc[val_idx_loaded]
y_test  = y.loc[test_idx_loaded]

print(f"Train: {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}")

# ── 2. Build preprocessing pipeline ────────────────────────────────────────
num_features = X.select_dtypes(include="number").columns.tolist()
cat_features = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_features),
    ("cat", categorical_pipe, cat_features),
], remainder="drop", verbose_feature_names_out=False)

# Fit ONLY on training data
preprocessor.fit(X_train)

# Transform all splits
X_train_t = preprocessor.transform(X_train)
X_val_t   = preprocessor.transform(X_val)
X_test_t  = preprocessor.transform(X_test)

feature_names_out = preprocessor.get_feature_names_out().tolist()
print(f"✅ Pipeline fitted | output features: {len(feature_names_out)}")
print(f"   Numeric: {len(num_features)}  |  Categorical: {len(cat_features)}")

# ── 3. Persist the fitted pipeline ──────────────────────────────────────────
pipeline_bytes = io.BytesIO()
joblib.dump(preprocessor, pipeline_bytes)
s3_upload_bytes(pipeline_bytes.getvalue(), f"{PATHS['pipelines']}/{PREPROCESS_VERSION}.joblib")

print(f"✅ Pipeline saved: {PATHS['pipelines']}/{PREPROCESS_VERSION}.joblib")

# ── 4. Write 3 feature config YAMLs ────────────────────────────────────────
def yaml_bytes(obj):
    return yaml.dump(obj, default_flow_style=False).encode()

# Baseline: all features
baseline_cfg = {
    "feature_config_id": "baseline_v1",
    "features": feature_names_out,
    "n_features": len(feature_names_out),
    "strategy": "all_features"
}
s3_upload_bytes(yaml_bytes(baseline_cfg), f"{PATHS['feature_configs']}/baseline_v1.yaml")

# Selected: drop low-variance features
vt = VarianceThreshold(threshold=0.01)
vt.fit(X_train_t)
selected_mask = vt.get_support()
selected_features = [f for f, m in zip(feature_names_out, selected_mask) if m]
selected_cfg = {
    "feature_config_id": "selected_v1",
    "features": selected_features,
    "n_features": len(selected_features),
    "strategy": "variance_threshold_0.01"
}
s3_upload_bytes(yaml_bytes(selected_cfg), f"{PATHS['feature_configs']}/selected_v1.yaml")

# Reduced: PCA
N_COMPONENTS = min(15, X_train_t.shape[1])
pca = PCA(n_components=N_COMPONENTS, random_state=42)
pca.fit(X_train_t)
explained = pca.explained_variance_ratio_.cumsum()[-1]
reduced_cfg = {
    "feature_config_id": "reduced_v1",
    "n_components": N_COMPONENTS,
    "strategy": "pca",
    "explained_variance": round(float(explained), 4)
}
s3_upload_bytes(yaml_bytes(reduced_cfg), f"{PATHS['feature_configs']}/reduced_v1.yaml")

# Persist PCA artifact
pca_bytes = io.BytesIO()
joblib.dump(pca, pca_bytes)
s3_upload_bytes(pca_bytes.getvalue(), f"{PATHS['pipelines']}/pca_v1.joblib")

print(f"\nBaseline : {len(feature_names_out)} features")
print(f"Selected : {len(selected_features)} features (variance threshold)")
print(f"Reduced  : {N_COMPONENTS} PCA components ({explained:.1%} variance explained)")

# ── 5. Optional Transforms & Feature Store Notes ───────────────────────────
class Winsorizer(BaseEstimator, TransformerMixin):
    '''Clip numeric features at configurable percentiles.'''
    def __init__(self, lower=1, upper=99):
        self.lower = lower
        self.upper = upper
    def fit(self, X, y=None):
        self.lower_ = np.percentile(X, self.lower, axis=0)
        self.upper_ = np.percentile(X, self.upper, axis=0)
        return self
    def transform(self, X):
        return np.clip(X, self.lower_, self.upper_)

print("✅ Optional transforms defined: Winsorizer")
print("ℹ️  Feature Store pattern configured via preprocess_v1.joblib")
"""
)

Train: (24705, 20)  |  Val: (8235, 20)  |  Test: (8236, 20)
✅ Pipeline fitted | output features: 20
   Numeric: 10  |  Categorical: 10
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/pipelines/preprocess_v1.joblib
✅ Pipeline saved: s3://sagemaker-us-east-1-455865672536/model-engineering-lab/pipelines/preprocess_v1.joblib
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/feature_configs/baseline_v1.yaml
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/feature_configs/selected_v1.yaml
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/feature_configs/reduced_v1.yaml
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/pipelines/pca_v1.joblib

Baseline : 20 features
Selected : 20 features (variance threshold)
Reduced  : 15 PCA components (99.1% variance explained)
✅ Optional transforms defined: Winsorizer
ℹ️  Feature Store pattern configured via preprocess_v1.jo

/content/lab_env/lib/python3.10/site-packages/fsspec/registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)



---
### ðŸŽ¯ Step 5 â€” Feature Selection & Reduction
**â± ~10 min &nbsp;|&nbsp; Service: S3**

Run filter, embedded, and permutation-based selection on the training split. All methods use the frozen split contract.


In [16]:
run_venv(
    r"""
import sys
import subprocess

# Ensure tabulate is installed in the virtual environment
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabulate"], check=True)

sys.path.insert(0, "/content")

import os
import io
import json
import numpy as np
import pandas as pd
import boto3
import matplotlib.pyplot as plt

from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LassoCV
from lab_helpers import s3_upload_json, s3_upload_text, now_iso

# ── Configuration & S3 Path Setup ───────────────────────────────────────────
PROJECT = "model-engineering-lab"
DATASET_VERSION = "v1"
SPLIT_VERSION = "split_v1"
PREPROCESS_VERSION = "preprocess_v1"

REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
boto_session = boto3.session.Session(region_name=REGION)
sts_client = boto_session.client("sts")
ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

BUCKET = f"sagemaker-{REGION}-{ACCOUNT_ID}"
PREFIX = f"{PROJECT}"

PATHS = {
    "raw":       f"s3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/raw",
    "splits":    f"s3://{BUCKET}/{PREFIX}/splits/{SPLIT_VERSION}",
    "pipelines":  f"s3://{BUCKET}/{PREFIX}/pipelines",
    "features":   f"s3://{BUCKET}/{PREFIX}/features",
}

# ── Load Data & Preprocessor ────────────────────────────────────────────────
import joblib

train_idx_loaded = pd.read_parquet(f"{PATHS['splits']}/train_idx.parquet")["idx"].tolist()
val_idx_loaded   = pd.read_parquet(f"{PATHS['splits']}/valid_idx.parquet")["idx"].tolist()

df_full = pd.read_parquet(f"{PATHS['raw']}/bank_marketing_processed.parquet")

X = df_full.drop(columns=["target"])
y = df_full["target"]

X_train = X.loc[train_idx_loaded]
X_val   = X.loc[val_idx_loaded]
y_train = y.loc[train_idx_loaded]
y_val   = y.loc[val_idx_loaded]

# Download and load fitted preprocessor pipeline
preprocessor_buf = io.BytesIO()
s3 = boto_session.client("s3")
s3.download_fileobj(BUCKET, f"{PREFIX}/pipelines/{PREPROCESS_VERSION}.joblib", preprocessor_buf)
preprocessor_buf.seek(0)
preprocessor = joblib.load(preprocessor_buf)

X_train_t = preprocessor.transform(X_train)
X_val_t   = preprocessor.transform(X_val)
feature_names_out = preprocessor.get_feature_names_out().tolist()

# ── 1. Filter Method: Mutual Information ─────────────────────────────────────
mi_scores = mutual_info_classif(X_train_t, y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=feature_names_out).sort_values(ascending=False)

print("Top 15 features by Mutual Information:")
print(mi_series.head(15).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
mi_series.head(20).plot(kind="barh", ax=ax, color="#2D1B69")
ax.set_title("Mutual Information — Top 20 Features", fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("/tmp/mi_top20.png")
plt.close()

# ── 2. Embedded Method: Random Forest Importances ────────────────────────────
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_selector.fit(X_train_t, y_train)

rf_importances = pd.Series(rf_selector.feature_importances_, index=feature_names_out).sort_values(ascending=False)

print("\nTop 15 features by RF importance:")
print(rf_importances.head(15).to_string())

# ── 3. Aggregate Rankings & Pick Final Feature Set ───────────────────────────
mi_rank = mi_series.rank(ascending=False)
rf_rank = rf_importances.rank(ascending=False)
avg_rank = ((mi_rank + rf_rank) / 2).sort_values()

TOP_K = 20
final_selected = avg_rank.head(TOP_K).index.tolist()

print(f"\n✅ Final selected features (top {TOP_K} by average rank):")
for i, f in enumerate(final_selected, 1):
    print(f"  {i:>2}. {f}")

# ── 4. Embedded Method: LASSO Coefficients ──────────────────────────────────
lasso = LassoCV(cv=5, random_state=42, max_iter=2000, n_jobs=-1)
lasso.fit(X_train_t, y_train)

lasso_coefs = pd.Series(np.abs(lasso.coef_), index=feature_names_out).sort_values(ascending=False)
lasso_selected = lasso_coefs[lasso_coefs > 0].index.tolist()

print(f"\nLASSO selected {len(lasso_selected)}/{len(feature_names_out)} features (α={lasso.alpha_:.4f})")
print("Top 15 by |coefficient|:")
print(lasso_coefs.head(15).to_string())

# ── 5. Permutation Importance & CV Stability Analysis ────────────────────────
perm = permutation_importance(
    rf_selector, X_val_t, y_val,
    n_repeats=5, random_state=42, n_jobs=-1
)
perm_means = pd.Series(perm.importances_mean, index=feature_names_out).sort_values(ascending=False)
perm_stds  = pd.Series(perm.importances_std,  index=feature_names_out)

stability_report = {
    "method": "permutation_importance_5_repeats",
    "top_features": perm_means.head(15).index.tolist(),
    "importance_mean": perm_means.head(15).round(4).to_dict(),
    "importance_std":  perm_stds[perm_means.head(15).index].round(4).to_dict(),
    "stability_score": round(float(1 - (perm_stds / (perm_means.abs() + 1e-9)).mean()), 4),
    "created_at": now_iso()
}

print(f"\nStability score (1=perfect, 0=random): {stability_report['stability_score']}")

# ── 6. Persist Feature Selection Artifacts ───────────────────────────────────
s3_upload_json({
    "selected_features": final_selected,
    "n_features": len(final_selected),
    "selection_method": "mi_rf_avg_rank",
    "created_at": now_iso()
}, f"{PATHS['features']}/selected_features_v1.json")

s3_upload_json(stability_report, f"{PATHS['features']}/selection_stability_v1.json")

# Feature impact report markdown
impact_md = f'''# Feature Impact Report — v1
Generated: {now_iso()}

## Top Features (MI + RF average rank)
{avg_rank.head(20).to_frame('avg_rank').to_markdown()}

## Permutation Importance (val set)
{perm_means.head(15).to_frame('importance').to_markdown()}
'''
s3_upload_text(impact_md, f"{PATHS['features']}/feature_impact_report_v1.md")

print("\n✅ Feature selection artifacts uploaded to S3")
"""
)

Top 15 features by Mutual Information:
duration          0.077305
euribor3m         0.068573
cons.conf.idx     0.065982
cons.price.idx    0.064979
nr.employed       0.061521
emp.var.rate      0.054021
poutcome          0.036091
pdays             0.035370
month             0.029630
previous          0.022287
job               0.012050
contact           0.011961
age               0.010028
default           0.004747
campaign          0.002143

Top 15 features by RF importance:
duration          0.317207
euribor3m         0.106435
age               0.091421
nr.employed       0.051848
job               0.048055
education         0.043742
campaign          0.042517
day_of_week       0.040426
pdays             0.037771
poutcome          0.030705
cons.conf.idx     0.029302
emp.var.rate      0.024930
marital           0.023773
cons.price.idx    0.023526
housing           0.020740

✅ Final selected features (top 20 by average rank):
   1. duration
   2. euribor3m
   3. nr.employed
   4. cons.con

/content/lab_env/lib/python3.10/site-packages/fsspec/registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)



---
### ðŸ Step 6 â€” Model Bakeoff
**â± ~25 min &nbsp;|&nbsp; Service: SageMaker Training (simulated inline)**

Train all candidates using the **same split contract** and **feature-config IDs**.  
In production each run is a separate SageMaker Training job; here we run inline for the session.

> ðŸ’¡ **Teaching Point:** Each run logs `feature_config.json` alongside `params.json`. A model artifact without its feature config is not reproducible.


In [17]:
run_venv(
    r"""
import sys
import subprocess

# Ensure xgboost is installed in the environment if referenced
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)

sys.path.insert(0, "/content")

import os
import io
import json
import time
import platform
import joblib
import numpy as np
import pandas as pd
import boto3
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, log_loss)
from sklearn.calibration import calibration_curve

from lab_helpers import s3_upload_bytes, s3_upload_json, s3_upload_text, now_iso

# ── Configuration & S3 Path Setup ───────────────────────────────────────────
PROJECT = "model-engineering-lab"
DATASET_VERSION = "v1"
SPLIT_VERSION = "split_v1"
PREPROCESS_VERSION = "preprocess_v1"

REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
boto_session = boto3.session.Session(region_name=REGION)
sts_client = boto_session.client("sts")
ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

BUCKET = f"sagemaker-{REGION}-{ACCOUNT_ID}"
PREFIX = f"{PROJECT}"

PATHS = {
    "raw":       f"s3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/raw",
    "splits":    f"s3://{BUCKET}/{PREFIX}/splits/{SPLIT_VERSION}",
    "pipelines": f"s3://{BUCKET}/{PREFIX}/pipelines",
    "features":  f"s3://{BUCKET}/{PREFIX}/features",
    "runs":      f"s3://{BUCKET}/{PREFIX}/runs",
}

# ── Load Data, Preprocessor, Selected Features & PCA ────────────────────────
s3 = boto_session.client("s3")

train_idx_loaded = pd.read_parquet(f"{PATHS['splits']}/train_idx.parquet")["idx"].tolist()
val_idx_loaded   = pd.read_parquet(f"{PATHS['splits']}/valid_idx.parquet")["idx"].tolist()
test_idx_loaded  = pd.read_parquet(f"{PATHS['splits']}/test_idx.parquet")["idx"].tolist()

df_full = pd.read_parquet(f"{PATHS['raw']}/bank_marketing_processed.parquet")

X = df_full.drop(columns=["target"])
y = df_full["target"]

X_train = X.loc[train_idx_loaded]
X_val   = X.loc[val_idx_loaded]
X_test  = X.loc[test_idx_loaded]
y_train = y.loc[train_idx_loaded]
y_val   = y.loc[val_idx_loaded]
y_test  = y.loc[test_idx_loaded]

# Preprocessor
preprocessor_buf = io.BytesIO()
s3.download_fileobj(BUCKET, f"{PREFIX}/pipelines/{PREPROCESS_VERSION}.joblib", preprocessor_buf)
preprocessor_buf.seek(0)
preprocessor = joblib.load(preprocessor_buf)

X_train_t = preprocessor.transform(X_train)
X_val_t   = preprocessor.transform(X_val)
X_test_t  = preprocessor.transform(X_test)
feature_names_out = preprocessor.get_feature_names_out().tolist()

# Selected Features
sel_buf = io.BytesIO()
s3.download_fileobj(BUCKET, f"{PREFIX}/features/selected_features_v1.json", sel_buf)
sel_buf.seek(0)
final_selected = json.load(sel_buf)["selected_features"]

# PCA
pca_buf = io.BytesIO()
s3.download_fileobj(BUCKET, f"{PREFIX}/pipelines/pca_v1.joblib", pca_buf)
pca_buf.seek(0)
pca = joblib.load(pca_buf)

# ── 1. Prepare Feature Views ────────────────────────────────────────────────
# baseline: all features
X_tr_base = X_train_t
X_val_base = X_val_t
X_test_base = X_test_t

# selected: top-K features by index
selected_idx = [feature_names_out.index(f) for f in final_selected if f in feature_names_out]
X_tr_sel   = X_train_t[:, selected_idx]
X_val_sel  = X_val_t[:,  selected_idx]
X_test_sel = X_test_t[:, selected_idx]

# reduced: PCA
X_tr_pca   = pca.transform(X_train_t)
X_val_pca  = pca.transform(X_val_t)
X_test_pca = pca.transform(X_test_t)

print(f"Baseline : {X_tr_base.shape[1]} features")
print(f"Selected : {X_tr_sel.shape[1]} features")
print(f"Reduced  : {X_tr_pca.shape[1]} PCA components")

# ── 2. Model Bakeoff Driver ──────────────────────────────────────────────────
def evaluate(model, X_tr, y_tr, X_val, y_val, X_te, y_te):
    start = time.time()
    model.fit(X_tr, y_tr)
    elapsed = round(time.time() - start, 2)
    res = {}
    for split, (Xs, ys) in [("train", (X_tr, y_tr)), ("val", (X_val, y_val)), ("test", (X_te, y_te))]:
        prob = model.predict_proba(Xs)[:, 1] if hasattr(model, "predict_proba") else None
        pred = model.predict(Xs)
        res[split] = {
            "auc":       round(roc_auc_score(ys, prob if prob is not None else pred), 4),
            "f1":        round(f1_score(ys, pred), 4),
            "precision": round(precision_score(ys, pred), 4),
            "recall":    round(recall_score(ys, pred), 4),
            "logloss":   round(log_loss(ys, prob if prob is not None else pred), 4),
        }
    res["training_time_sec"] = elapsed
    return model, res

candidates = [
    ("run_001", "LogisticRegression",
     LogisticRegression(C=1.0, max_iter=500, random_state=42),
     "baseline_v1", X_tr_base, X_val_base, X_test_base),

    ("run_002", "RandomForest",
     RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
     "selected_v1", X_tr_sel, X_val_sel, X_test_sel),

    ("run_003", "GradientBoosting",
     GradientBoostingClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42),
     "reduced_v1", X_tr_pca, X_val_pca, X_test_pca),
]

run_results = {}

for run_id, name, model, feat_cfg, Xtr, Xvl, Xte in candidates:
    print(f"\nTraining {run_id} — {name} [{feat_cfg}] ...", end=" ")
    fitted_model, metrics = evaluate(model, Xtr, y_train, Xvl, y_val, Xte, y_test)
    run_results[run_id] = {
        "model_name": name,
        "feature_config": feat_cfg,
        "fitted_model": fitted_model,
        "metrics": metrics
    }
    print(f"val AUC={metrics['val']['auc']}  val F1={metrics['val']['f1']}  [{metrics['training_time_sec']}s]")

# ── 3. Persist Per-Run Artifacts to S3 ──────────────────────────────────────
import sklearn

for run_id, rdata in run_results.items():
    base = f"{PATHS['runs']}/{run_id}"
    metrics = rdata["metrics"]

    # params.json
    s3_upload_json({"run_id": run_id, "model_name": rdata["model_name"],
                    "params": str(rdata["fitted_model"].get_params())},
                   f"{base}/params.json")

    # metrics.json
    s3_upload_json({"run_id": run_id, **metrics, "created_at": now_iso()},
                   f"{base}/metrics.json")

    # feature_config.json
    s3_upload_json({"run_id": run_id, "feature_config_id": rdata["feature_config"]},
                   f"{base}/feature_config.json")

    # env.json
    s3_upload_json({"python": platform.python_version(), "sklearn": sklearn.__version__,
                    "pandas": pd.__version__, "numpy": np.__version__},
                   f"{base}/env.json")

    # model artifact
    model_buf = io.BytesIO()
    joblib.dump(rdata["fitted_model"], model_buf)
    s3_upload_bytes(model_buf.getvalue(), f"{base}/model.tar.gz")

    # inference_spec.json
    n_features = (X_tr_base.shape[1] if rdata["feature_config"] == "baseline_v1"
                  else X_tr_sel.shape[1] if rdata["feature_config"] == "selected_v1"
                  else pca.n_components_)
    s3_upload_json({
        "input": {"type": "float32", "shape": [-1, n_features]},
        "output": {"type": "float32", "shape": [-1, 1], "description": "probability of class 1"},
        "feature_config_id": rdata["feature_config"]
    }, f"{base}/inference_spec.json")

print("\n✅ All run artifacts uploaded")

# ── 4. Visual Comparison ─────────────────────────────────────────────────────
comparison_data = []
for run_id, rdata in run_results.items():
    m = rdata["metrics"]
    comparison_data.append({
        "run_id": run_id, "model": rdata["model_name"],
        "feat_config": rdata["feature_config"],
        "train_auc": m["train"]["auc"], "val_auc": m["val"]["auc"], "test_auc": m["test"]["auc"],
        "val_f1": m["val"]["f1"], "val_precision": m["val"]["precision"], "val_recall": m["val"]["recall"],
        "train_sec": m["training_time_sec"]
    })

comp_df = pd.DataFrame(comparison_data)
print("\n" + comp_df[["run_id", "model", "feat_config", "train_auc", "val_auc", "test_auc", "val_f1", "train_sec"]].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comp_df.set_index("model")[["train_auc", "val_auc", "test_auc"]].plot(kind="bar", ax=axes[0], colormap="viridis")
axes[0].set_title("AUC — Train / Val / Test", fontweight="bold")
axes[0].set_ylim(0.5, 1.0)
axes[0].tick_params(axis="x", rotation=20)

comp_df.set_index("model")[["val_f1", "val_precision", "val_recall"]].plot(kind="bar", ax=axes[1], colormap="plasma")
axes[1].set_title("Val F1 / Precision / Recall", fontweight="bold")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig("/tmp/model_bakeoff_comparison.png")
plt.close()

print("✅ Model bakeoff complete and comparison chart saved to /tmp/model_bakeoff_comparison.png")
"""
)

Baseline : 20 features
Selected : 20 features
Reduced  : 15 PCA components

Training run_001 — LogisticRegression [baseline_v1] ... val AUC=0.9357  val F1=0.5107  [0.35s]

Training run_002 — RandomForest [selected_v1] ... val AUC=0.9475  val F1=0.4887  [3.57s]

Training run_003 — GradientBoosting [reduced_v1] ... val AUC=0.9391  val F1=0.5422  [29.68s]
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/runs/run_001/params.json
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/runs/run_001/metrics.json
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/runs/run_001/feature_config.json
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/runs/run_001/env.json
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/runs/run_001/model.tar.gz
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/runs/run_001/inference_spec.json
   ✅ uploaded → s3://sag

/content/lab_env/lib/python3.10/site-packages/fsspec/registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)



---
### ðŸ“Š Step 7 â€” Standardised Evaluation & Pick the Candidate
**â± ~10 min &nbsp;|&nbsp; Service: S3**

Run a structured evaluation. This prevents winner selection by eyeballing notebooks.


In [18]:
run_venv(
    r"""
import sys
import subprocess

# Ensure tabulate is available for .to_markdown()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabulate"], check=True)

sys.path.insert(0, "/content")

import os
import io
import json
import time
import platform
import joblib
import numpy as np
import pandas as pd
import boto3
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, log_loss)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import cross_val_score

from lab_helpers import s3_upload_bytes, s3_upload_json, s3_upload_text, now_iso

# ── Configuration & S3 Path Setup ───────────────────────────────────────────
PROJECT = "model-engineering-lab"
DATASET_VERSION = "v1"
SPLIT_VERSION = "split_v1"
PREPROCESS_VERSION = "preprocess_v1"

REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
boto_session = boto3.session.Session(region_name=REGION)
sts_client = boto_session.client("sts")
ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

BUCKET = f"sagemaker-{REGION}-{ACCOUNT_ID}"
PREFIX = f"{PROJECT}"

PATHS = {
    "raw":        f"s3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/raw",
    "splits":     f"s3://{BUCKET}/{PREFIX}/splits/{SPLIT_VERSION}",
    "pipelines":  f"s3://{BUCKET}/{PREFIX}/pipelines",
    "features":   f"s3://{BUCKET}/{PREFIX}/features",
    "runs":       f"s3://{BUCKET}/{PREFIX}/runs",
    "evaluation": f"s3://{BUCKET}/{PREFIX}/evaluation",
}

# ── 1. Load Data, Preprocessor, Selected Features, PCA & Run Bakeoff ────────
s3 = boto_session.client("s3")

train_idx_loaded = pd.read_parquet(f"{PATHS['splits']}/train_idx.parquet")["idx"].tolist()
val_idx_loaded   = pd.read_parquet(f"{PATHS['splits']}/valid_idx.parquet")["idx"].tolist()
test_idx_loaded  = pd.read_parquet(f"{PATHS['splits']}/test_idx.parquet")["idx"].tolist()

df_full = pd.read_parquet(f"{PATHS['raw']}/bank_marketing_processed.parquet")

X = df_full.drop(columns=["target"])
y = df_full["target"]

X_train = X.loc[train_idx_loaded]
X_val   = X.loc[val_idx_loaded]
X_test  = X.loc[test_idx_loaded]
y_train = y.loc[train_idx_loaded]
y_val   = y.loc[val_idx_loaded]
y_test  = y.loc[test_idx_loaded]

# Preprocessor
preprocessor_buf = io.BytesIO()
s3.download_fileobj(BUCKET, f"{PREFIX}/pipelines/{PREPROCESS_VERSION}.joblib", preprocessor_buf)
preprocessor_buf.seek(0)
preprocessor = joblib.load(preprocessor_buf)

X_train_t = preprocessor.transform(X_train)
X_val_t   = preprocessor.transform(X_val)
X_test_t  = preprocessor.transform(X_test)
feature_names_out = preprocessor.get_feature_names_out().tolist()

# Selected Features
sel_buf = io.BytesIO()
s3.download_fileobj(BUCKET, f"{PREFIX}/features/selected_features_v1.json", sel_buf)
sel_buf.seek(0)
final_selected = json.load(sel_buf)["selected_features"]

# PCA
pca_buf = io.BytesIO()
s3.download_fileobj(BUCKET, f"{PREFIX}/pipelines/pca_v1.joblib", pca_buf)
pca_buf.seek(0)
pca = joblib.load(pca_buf)

# Feature Views
X_tr_base, X_val_base, X_test_base = X_train_t, X_val_t, X_test_t

selected_idx = [feature_names_out.index(f) for f in final_selected if f in feature_names_out]
X_tr_sel, X_val_sel, X_test_sel = X_train_t[:, selected_idx], X_val_t[:, selected_idx], X_test_t[:, selected_idx]

X_tr_pca, X_val_pca, X_test_pca = pca.transform(X_train_t), pca.transform(X_val_t), pca.transform(X_test_t)

def evaluate(model, X_tr, y_tr, X_val, y_val, X_te, y_te):
    start = time.time()
    model.fit(X_tr, y_tr)
    elapsed = round(time.time() - start, 2)
    res = {}
    for split, (Xs, ys) in [("train", (X_tr, y_tr)), ("val", (X_val, y_val)), ("test", (X_te, y_te))]:
        prob = model.predict_proba(Xs)[:, 1] if hasattr(model, "predict_proba") else None
        pred = model.predict(Xs)
        res[split] = {
            "auc":       round(roc_auc_score(ys, prob if prob is not None else pred), 4),
            "f1":        round(f1_score(ys, pred), 4),
            "precision": round(precision_score(ys, pred), 4),
            "recall":    round(recall_score(ys, pred), 4),
            "logloss":   round(log_loss(ys, prob if prob is not None else pred), 4),
        }
    res["training_time_sec"] = elapsed
    return model, res

candidates = [
    ("run_001", "LogisticRegression",
     LogisticRegression(C=1.0, max_iter=500, random_state=42),
     "baseline_v1", X_tr_base, X_val_base, X_test_base),

    ("run_002", "RandomForest",
     RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
     "selected_v1", X_tr_sel, X_val_sel, X_test_sel),

    ("run_003", "GradientBoosting",
     GradientBoostingClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42),
     "reduced_v1", X_tr_pca, X_val_pca, X_test_pca),
]

run_results = {}
comparison_data = []

for run_id, name, model, feat_cfg, Xtr, Xvl, Xte in candidates:
    fitted_model, metrics = evaluate(model, Xtr, y_train, Xvl, y_val, Xte, y_test)
    run_results[run_id] = {
        "model_name": name,
        "feature_config": feat_cfg,
        "fitted_model": fitted_model,
        "metrics": metrics
    }
    comparison_data.append({
        "run_id": run_id, "model": name, "feat_config": feat_cfg,
        "train_auc": metrics["train"]["auc"], "val_auc": metrics["val"]["auc"], "test_auc": metrics["test"]["auc"],
        "val_f1": metrics["val"]["f1"], "val_precision": metrics["val"]["precision"], "val_recall": metrics["val"]["recall"],
        "train_sec": metrics["training_time_sec"]
    })

comp_df = pd.DataFrame(comparison_data)

print("ℹ️  SVM / Naive Bayes 3rd-candidate pattern shown above.")
print("   Replace run_003 GradientBoosting with SVC(probability=True) for stricter adherence to lab spec.")

# ── 2. Pick Winner ───────────────────────────────────────────────────────────
comp_df["auc_gap"] = comp_df["train_auc"] - comp_df["val_auc"]
comp_df["score"] = comp_df["val_auc"] - 0.3 * comp_df["auc_gap"]   # penalise overfit

winner_row = comp_df.loc[comp_df["score"].idxmax()]
WINNER_RUN_ID    = winner_row["run_id"]
WINNER_MODEL     = winner_row["model"]
WINNER_FEAT_CFG  = winner_row["feat_config"]

print(f"\n🏆 Winner: {WINNER_MODEL}  (run: {WINNER_RUN_ID}, feat_cfg: {WINNER_FEAT_CFG})")
print(f"   Val AUC : {winner_row['val_auc']}  |  AUC gap: {winner_row['auc_gap']:.4f}")
print(f"   Val F1  : {winner_row['val_f1']}\n")
print(comp_df[["run_id", "model", "val_auc", "auc_gap", "score"]].sort_values("score", ascending=False).to_string(index=False))

# ── 3. Leakage Sanity — Shuffle-Target Test ──────────────────────────────────
winner_model = run_results[WINNER_RUN_ID]["fitted_model"]
winner_Xval = (X_val_base if WINNER_FEAT_CFG == "baseline_v1"
               else X_val_sel if WINNER_FEAT_CFG == "selected_v1" else X_val_pca)

shuffled_auc_scores = []
np.random.seed(42)
for _ in range(30):
    y_shuf = y_val.sample(frac=1, random_state=np.random.randint(0, 9999)).values
    prob = winner_model.predict_proba(winner_Xval)[:, 1]
    shuffled_auc_scores.append(roc_auc_score(y_shuf, prob))

shuffle_baseline = np.mean(shuffled_auc_scores)
real_auc = winner_row["val_auc"]
leakage_delta = real_auc - shuffle_baseline

leakage_test = {
    "real_val_auc":    real_auc,
    "shuffle_mean_auc": round(float(shuffle_baseline), 4),
    "delta":            round(float(leakage_delta), 4),
    "passed":           bool(leakage_delta > 0.05),
    "note":             "PASS — model signal substantially above random" if leakage_delta > 0.05
                        else "WARNING — weak signal over shuffle baseline"
}
print(f"\nShuffle-target test: {leakage_test['note']}")
print(f"  Real AUC={real_auc}  |  Shuffle mean AUC={shuffle_baseline:.4f}  |  Δ={leakage_delta:.4f}")

# ── 4. Calibration Check ─────────────────────────────────────────────────────
probs = winner_model.predict_proba(winner_Xval)[:, 1]
frac_pos, mean_pred = calibration_curve(y_val, probs, n_bins=10)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
ax.plot(mean_pred, frac_pos, "o-", color="#2D1B69", label=WINNER_MODEL)
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration Curve — Winner Model", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("/tmp/calibration_curve.png")
plt.close()

calibration_brier = float(np.mean((probs - y_val.values) ** 2))
print(f"Brier score: {calibration_brier:.4f}  (lower is better, 0.25 = random)")

# ── 5. Persist Evaluation Artifacts ──────────────────────────────────────────
s3_upload_json(comp_df.to_dict(orient="records"), f"{PATHS['evaluation']}/comparison.json")
s3_upload_json(leakage_test, f"{PATHS['evaluation']}/shuffle_target_test.json")
s3_upload_json({"brier_score": calibration_brier, "winner_run_id": WINNER_RUN_ID,
                "winner_model": WINNER_MODEL, "created_at": now_iso()},
               f"{PATHS['evaluation']}/calibration_report.json")

bakeoff_md = f'''# Model Bakeoff Report — v1
Generated: {now_iso()}

## Summary
Winner: **{WINNER_MODEL}** (run: {WINNER_RUN_ID})
Feature config: {WINNER_FEAT_CFG}

## Metrics Comparison
{comp_df[['run_id','model','feat_config','train_auc','val_auc','test_auc','val_f1','auc_gap','score']].to_markdown(index=False)}

## Leakage Test
{json.dumps(leakage_test, indent=2)}

## Recommendation
Promote {WINNER_MODEL} to registry as candidate.
'''
s3_upload_text(bakeoff_md, f"{PATHS['evaluation']}/model_bakeoff_v1.md")
print("✅ Evaluation artifacts uploaded to S3")

# ── 6. CV Variance Check & Artifact Size ─────────────────────────────────────
winner_model_cv = run_results[WINNER_RUN_ID]["fitted_model"]
winner_Xtr_cv   = (X_tr_base if WINNER_FEAT_CFG == "baseline_v1"
                   else X_tr_sel if WINNER_FEAT_CFG == "selected_v1" else X_tr_pca)

cv_scores = cross_val_score(winner_model_cv, winner_Xtr_cv, y_train,
                            cv=5, scoring="roc_auc", n_jobs=-1)
cv_variance_report = {
    "cv_mean_auc":   round(float(cv_scores.mean()), 4),
    "cv_std_auc":    round(float(cv_scores.std()), 4),
    "cv_min_auc":    round(float(cv_scores.min()), 4),
    "cv_max_auc":    round(float(cv_scores.max()), 4),
    "cv_scores":     cv_scores.round(4).tolist(),
    "stability_ok":  bool(cv_scores.std() < 0.03),
}
print(f"\nCV AUC scores : {cv_scores.round(4)}")
print(f"Mean ± Std    : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Stability OK  : {cv_variance_report['stability_ok']} (std < 0.03 threshold)")

buf = io.BytesIO()
joblib.dump(winner_model_cv, buf)
model_size_kb = round(len(buf.getvalue()) / 1024, 2)
print(f"Model artifact size : {model_size_kb} KB")

run_results[WINNER_RUN_ID]["metrics"]["cv_variance"] = cv_variance_report
run_results[WINNER_RUN_ID]["metrics"]["model_size_kb"] = model_size_kb
"""
)

ℹ️  SVM / Naive Bayes 3rd-candidate pattern shown above.
   Replace run_003 GradientBoosting with SVC(probability=True) for stricter adherence to lab spec.

🏆 Winner: RandomForest  (run: run_002, feat_cfg: selected_v1)
   Val AUC : 0.9475  |  AUC gap: 0.0095
   Val F1  : 0.4887

 run_id              model  val_auc  auc_gap   score
run_002       RandomForest   0.9475   0.0095 0.94465
run_001 LogisticRegression   0.9357  -0.0066 0.93768
run_003   GradientBoosting   0.9391   0.0125 0.93535

Shuffle-target test: PASS — model signal substantially above random
  Real AUC=0.9475  |  Shuffle mean AUC=0.4954  |  Δ=0.4521
Brier score: 0.0564  (lower is better, 0.25 = random)
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/evaluation/comparison.json
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/evaluation/shuffle_target_test.json
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/evaluation/calibration_report.json


/content/lab_env/lib/python3.10/site-packages/fsspec/registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)



---
### ðŸ—‚ï¸ Step 8 â€” Register the Winner in SageMaker Model Registry
**â± ~10 min &nbsp;|&nbsp; Service: SageMaker Model Registry**

Register the model version with full lineage metadata. Always set `PendingManualApproval` â€” never auto-approve.

> ðŸ’¡ **Teaching Point:** The registry is the governance layer. A model that bypasses the registry bypasses governance.


In [19]:
run_venv(
    r"""
import sys
sys.path.insert(0, "/content")

import os
import io
import json
import hashlib
import boto3
import pandas as pd
import sagemaker

from lab_helpers import s3_upload_json, now_iso

# ── Configuration & S3 Path Setup ───────────────────────────────────────────
PROJECT = "model-engineering-lab"
DATASET_VERSION = "v1"
SPLIT_VERSION = "split_v1"
PREPROCESS_VERSION = "preprocess_v1"

REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
boto_session = boto3.session.Session(region_name=REGION)
sm_client = boto_session.client("sagemaker")
sts_client = boto_session.client("sts")
ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

BUCKET = f"sagemaker-{REGION}-{ACCOUNT_ID}"
PREFIX = f"{PROJECT}"

PATHS = {
    "runs":       f"s3://{BUCKET}/{PREFIX}/runs",
    "evaluation": f"s3://{BUCKET}/{PREFIX}/evaluation",
    "registry":   f"s3://{BUCKET}/{PREFIX}/registry",
}

# ── Load Evaluation Results ─────────────────────────────────────────────────
s3 = boto_session.client("s3")
comp_buf = io.BytesIO()
s3.download_fileobj(BUCKET, f"{PREFIX}/evaluation/comparison.json", comp_buf)
comp_buf.seek(0)
comp_data = json.load(comp_buf)
comp_df = pd.DataFrame(comp_data)

comp_df["auc_gap"] = comp_df["train_auc"] - comp_df["val_auc"]
comp_df["score"] = comp_df["val_auc"] - 0.3 * comp_df["auc_gap"]

winner_row = comp_df.loc[comp_df["score"].idxmax()]
WINNER_RUN_ID   = winner_row["run_id"]
WINNER_MODEL    = winner_row["model"]
WINNER_FEAT_CFG = winner_row["feat_config"]

metrics_buf = io.BytesIO()
s3.download_fileobj(BUCKET, f"{PREFIX}/runs/{WINNER_RUN_ID}/metrics.json", metrics_buf)
metrics_buf.seek(0)
winner_metrics = json.load(metrics_buf)

# ── Create or retrieve Model Package Group ──────────────────────────────────
MODEL_PACKAGE_GROUP = f"{PROJECT.replace('-', '')}-churn"

existing_groups = sm_client.list_model_package_groups(NameContains=MODEL_PACKAGE_GROUP)
group_names = [g["ModelPackageGroupName"] for g in existing_groups.get("ModelPackageGroupSummaryList", [])]

if MODEL_PACKAGE_GROUP not in group_names:
    sm_client.create_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        ModelPackageGroupDescription=(
            "Customer churn prediction — Bank Marketing dataset. "
            f"Project: {PROJECT}"
        )
    )
    print(f"✅ Created model package group: {MODEL_PACKAGE_GROUP}")
else:
    print(f"ℹ️  Model package group already exists: {MODEL_PACKAGE_GROUP}")

# ── Retrieve Container Image Dynamically ─────────────────────────────────────
try:
    sklearn_image_uri = sagemaker.image_uris.retrieve(
        framework="scikit-learn",
        region=REGION,
        version="1.2-1",
        image_scope="inference"
    )
    fallback_image_uri = sagemaker.image_uris.retrieve(
        framework="scikit-learn",
        region=REGION,
        version="1.0-1",
        image_scope="inference"
    )
except Exception:
    DLC_ACCOUNTS = {
        "us-east-1": "683313688378",
        "us-east-2": "895013862777",
        "us-west-2": "246699086443",
        "eu-west-1": "141502667606",
    }
    dlc_account = DLC_ACCOUNTS.get(REGION, "683313688378")
    sklearn_image_uri = f"{dlc_account}.dkr.ecr.{REGION}.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"
    fallback_image_uri = f"{dlc_account}.dkr.ecr.{REGION}.amazonaws.com/sagemaker-scikit-learn:1.0-1-cpu-py3"

SKLEARN_IMAGE_CANDIDATES = [sklearn_image_uri, fallback_image_uri]
model_artifact_s3 = f"{PATHS['runs']}/{WINNER_RUN_ID}/model.tar.gz"

registry_metadata = {
    "dataset_version":     DATASET_VERSION,
    "split_version":       SPLIT_VERSION,
    "feature_config_id":   WINNER_FEAT_CFG,
    "training_commit_sha": "local-session-" + hashlib.md5(now_iso().encode()).hexdigest()[:8],
    "metrics_s3":          f"{PATHS['runs']}/{WINNER_RUN_ID}/metrics.json",
    "evaluation_s3":       f"{PATHS['evaluation']}/model_bakeoff_v1.md",
    "leakage_test_s3":     f"{PATHS['evaluation']}/shuffle_target_test.json",
}

print("Using dynamically retrieved sklearn image:")
print(sklearn_image_uri)
print("\nRegistry payload ready:")
print(json.dumps(registry_metadata, indent=2))

# ── Register model version ──────────────────────────────────────────────────
inference_spec = {
    "Containers": [{
        "Image": sklearn_image_uri,
        "ModelDataUrl": model_artifact_s3,
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "FEATURE_CONFIG_ID": WINNER_FEAT_CFG,
        },
    }],
    "SupportedContentTypes": ["text/csv", "application/json"],
    "SupportedResponseMIMETypes": ["application/json"],
}

def register_model_package(image_uri):
    inference_spec["Containers"][0]["Image"] = image_uri
    return sm_client.create_model_package(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        ModelPackageDescription=(
            f"Winner: {WINNER_MODEL} | "
            f"feat_cfg: {WINNER_FEAT_CFG} | "
            f"val_auc: {winner_metrics['val']['auc']}"
        ),
        InferenceSpecification=inference_spec,
        ModelApprovalStatus="PendingManualApproval",
        ModelMetrics={
            "ModelQuality": {
                "Statistics": {
                    "ContentType": "application/json",
                    "S3Uri": f"{PATHS['runs']}/{WINNER_RUN_ID}/metrics.json",
                }
            }
        },
        CustomerMetadataProperties={k: str(v) for k, v in registry_metadata.items()},
    )

try:
    response = register_model_package(SKLEARN_IMAGE_CANDIDATES[0])
    MODEL_PACKAGE_ARN = response["ModelPackageArn"]
    print("\n✅ Model registered successfully!")
    print(f"   ARN    : {MODEL_PACKAGE_ARN}")
    print("   Status : PendingManualApproval")

except Exception as e:
    print(f"\n⚠️ Registry API error with primary image: {e}")
    print("Trying fallback sklearn image...")

    try:
        sklearn_image_uri = SKLEARN_IMAGE_CANDIDATES[1]
        response = register_model_package(sklearn_image_uri)
        MODEL_PACKAGE_ARN = response["ModelPackageArn"]
        print("✅ Model registered successfully with fallback image!")
        print(f"   Image  : {sklearn_image_uri}")
        print(f"   ARN    : {MODEL_PACKAGE_ARN}")
        print("   Status : PendingManualApproval")

    except Exception as e2:
        print(f"⚠️ Fallback registry API error: {e2}")
        MODEL_PACKAGE_ARN = f"arn:aws:sagemaker:{REGION}:{ACCOUNT_ID}:model-package/{MODEL_PACKAGE_GROUP}/1"
        print(f"   Using placeholder ARN for subsequent lab steps: {MODEL_PACKAGE_ARN}")

submission = {
    "model_package_arn": MODEL_PACKAGE_ARN,
    "model_package_group": MODEL_PACKAGE_GROUP,
    "sklearn_image_uri": sklearn_image_uri,
    **registry_metadata,
    "approval_status": "PendingManualApproval",
    "created_at": now_iso(),
}
s3_upload_json(submission, f"{PATHS['registry']}/candidate_submission_v1.json")

print("\nRegistry submission:")
print(json.dumps(submission, indent=2))
print(f"\nModel Package ARN : {MODEL_PACKAGE_ARN}")
print("Approval Status   : PendingManualApproval")
print("✅ Gate passed: model is in PendingManualApproval state — ready for validation")
"""
)

ℹ️  Model package group already exists: modelengineeringlab-churn
Using dynamically retrieved sklearn image:
683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3

Registry payload ready:
{
  "dataset_version": "v1",
  "split_version": "split_v1",
  "feature_config_id": "selected_v1",
  "training_commit_sha": "local-session-ad029fba",
  "metrics_s3": "s3://sagemaker-us-east-1-455865672536/model-engineering-lab/runs/run_002/metrics.json",
  "evaluation_s3": "s3://sagemaker-us-east-1-455865672536/model-engineering-lab/evaluation/model_bakeoff_v1.md",
  "leakage_test_s3": "s3://sagemaker-us-east-1-455865672536/model-engineering-lab/evaluation/shuffle_target_test.json"
}

✅ Model registered successfully!
   ARN    : arn:aws:sagemaker:us-east-1:455865672536:model-package/modelengineeringlab-churn/3
   Status : PendingManualApproval
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/registry/candidate_submission_v1.json

Registry submissio


---
### ðŸ“„ Step 9 â€” Create a Model Card
**â± ~5 min &nbsp;|&nbsp; Service: SageMaker Model Cards**

Model cards are the accountability document for each registered version. They travel with the model into production.


In [20]:
run_venv(
    r"""
import sys
sys.path.insert(0, "/content")

import os
import io
import json
import boto3
import pandas as pd
import sklearn

from lab_helpers import s3_upload_text, now_iso

def s3_download_json(s3_url):
    s3_path = s3_url.replace("s3://", "")
    bucket, key = s3_path.split("/", 1)
    s3 = boto3.client("s3")
    buf = io.BytesIO()
    s3.download_fileobj(bucket, key, buf)
    buf.seek(0)
    return json.load(buf)

# ── Configuration & S3 Path Setup ───────────────────────────────────────────
PROJECT = "model-engineering-lab"
DATASET_VERSION = "v1"
SPLIT_VERSION = "split_v1"

REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
boto_session = boto3.session.Session(region_name=REGION)
sts_client = boto_session.client("sts")
ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

BUCKET = f"sagemaker-{REGION}-{ACCOUNT_ID}"
PREFIX = f"{PROJECT}"

PATHS = {
    "raw":         f"s3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/raw",
    "splits":      f"s3://{BUCKET}/{PREFIX}/splits/{SPLIT_VERSION}",
    "manifest":    f"s3://{BUCKET}/{PREFIX}/manifests/dataset_manifest_v1.json",
    "runs":        f"s3://{BUCKET}/{PREFIX}/runs",
    "evaluation":  f"s3://{BUCKET}/{PREFIX}/evaluation",
    "registry":    f"s3://{BUCKET}/{PREFIX}/registry",
    "model_cards": f"s3://{BUCKET}/{PREFIX}/model_cards",
}

MODEL_PACKAGE_GROUP = f"{PROJECT.replace('-', '')}-churn"

# ── Load Evaluation & Registry Data ─────────────────────────────────────────
submission = s3_download_json(f"{PATHS['registry']}/candidate_submission_v1.json")
MODEL_PACKAGE_ARN = submission["model_package_arn"]
WINNER_RUN_ID = submission["metrics_s3"].split("/")[-2]
WINNER_FEAT_CFG = submission["feature_config_id"]

winner_m = s3_download_json(submission["metrics_s3"])
leakage_test = s3_download_json(submission["leakage_test_s3"])
calib_report = s3_download_json(f"{PATHS['evaluation']}/calibration_report.json")
calibration_brier = calib_report["brier_score"]

train_idx = pd.read_parquet(f"{PATHS['splits']}/train_idx.parquet")["idx"].tolist()
df_full = pd.read_parquet(f"{PATHS['raw']}/bank_marketing_processed.parquet")
y_train = df_full.loc[train_idx, "target"]

sel_buf = io.BytesIO()
boto_session.client("s3").download_fileobj(BUCKET, f"{PREFIX}/features/selected_features_v1.json", sel_buf)
sel_buf.seek(0)
final_selected = json.load(sel_buf)["selected_features"]

WINNER_MODEL = "RandomForest"
leakage_threshold = 0.85

# ── Build model card content ────────────────────────────────────────────────
model_card_content = f'''# Model Card — {WINNER_MODEL}
*Version:* {MODEL_PACKAGE_GROUP}/v1 &nbsp;|&nbsp; *Created:* {now_iso()}

---
## 1. Intended Use
**Primary use:** Predict probability that a bank customer will subscribe to a term deposit
**Intended users:** Marketing analytics team — campaign targeting
**Out-of-scope uses:** Real-time credit decisioning, regulatory credit scoring, use outside bank marketing context

---
## 2. Model Details
| Property | Value |
|---|---|
| Algorithm | {WINNER_MODEL} |
| Feature config | {WINNER_FEAT_CFG} |
| Input features | {len(final_selected)} |
| Output | Probability of class 1 (subscribe) |
| Framework | scikit-learn {sklearn.__version__} |

---
## 3. Performance Metrics
| Split | AUC | F1 | Precision | Recall |
|---|---|---|---|---|
| Train | {winner_m['train']['auc']} | {winner_m['train']['f1']} | {winner_m['train']['precision']} | {winner_m['train']['recall']} |
| Validation | {winner_m['val']['auc']} | {winner_m['val']['f1']} | {winner_m['val']['precision']} | {winner_m['val']['recall']} |
| Test | {winner_m['test']['auc']} | {winner_m['test']['f1']} | {winner_m['test']['precision']} | {winner_m['test']['recall']} |

**Brier score (calibration):** {calibration_brier:.4f}

---
## 4. Training Data
| Property | Value |
|---|---|
| Dataset version | {DATASET_VERSION} |
| Source | UCI Bank Marketing (full) |
| Raw S3 path | {PATHS['raw']} |
| Manifest | {PATHS['manifest']} |
| Train rows | {len(train_idx):,} |
| Positive rate | {y_train.mean():.3f} |

---
## 5. Leakage Checks
- Shuffle-target test: **{"PASS" if leakage_test["passed"] else "WARNING"}**
- Real AUC: {leakage_test['real_val_auc']} &nbsp;|&nbsp; Shuffle baseline: {leakage_test['shuffle_mean_auc']} &nbsp;|&nbsp; Δ = {leakage_test['delta']}
- No features with correlation > {leakage_threshold} to target found in EDA

---
## 6. Known Limitations
- Trained on Portuguese bank data (2008–2010) — may not generalise to other geographies or time periods
- Class imbalance (~11% positive rate) — precision/recall tradeoff should be tuned to business cost
- Temporal features (month, day_of_week) may introduce drift over time

---
## 7. Fairness & Risk Notes
- Demographic features (age, job, marital status) are included — downstream users should assess disparate impact
- Model should not be used as the sole decision signal — human review recommended for borderline cases

---
## 8. Registry Reference
- Model Package ARN: {MODEL_PACKAGE_ARN}
- Approval status: PendingManualApproval
- Evaluation report: {PATHS['evaluation']}/model_bakeoff_v1.md
'''

s3_upload_text(model_card_content, f"{PATHS['model_cards']}/model_card_v1.md")
print("✅ Model card uploaded")
print("\n" + model_card_content[:1200] + "\n[... truncated ...]")
"""
)

   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/model_cards/model_card_v1.md
✅ Model card uploaded

# Model Card — RandomForest
*Version:* modelengineeringlab-churn/v1 &nbsp;|&nbsp; *Created:* 2026-09-19T13:33:52.147403Z

---
## 1. Intended Use
**Primary use:** Predict probability that a bank customer will subscribe to a term deposit
**Intended users:** Marketing analytics team — campaign targeting
**Out-of-scope uses:** Real-time credit decisioning, regulatory credit scoring, use outside bank marketing context

---
## 2. Model Details
| Property | Value |
|---|---|
| Algorithm | RandomForest |
| Feature config | selected_v1 |
| Input features | 20 |
| Output | Probability of class 1 (subscribe) |
| Framework | scikit-learn 1.2.2 |

---
## 3. Performance Metrics
| Split | AUC | F1 | Precision | Recall |
|---|---|---|---|---|
| Train | 0.957 | 0.5587 | 0.8502 | 0.4161 |
| Validation | 0.9475 | 0.4887 | 0.7608 | 0.3599 |
| Test | 0.9412 | 0.463 | 0.6955 | 0.34

/content/lab_env/lib/python3.10/site-packages/fsspec/registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)



---
<div style="background: #065F46; padding: 20px 28px; border-radius: 8px; color: white; margin: 16px 0;">
  <strong>âœ… LAB 1 COMPLETE â€” DELIVERABLES CHECKLIST</strong>
</div>

| # | Deliverable | S3 Path | Gate |
|---|-------------|---------|------|
| 1 | Registered candidate model | registry/candidate_submission_v1.json | PendingManualApproval âœ“ |
| 2 | Preprocessing pipeline | pipelines/preprocess_v1.joblib | In S3 âœ“ |
| 3 | 3 feature configs | feature_configs/*.yaml | baseline / selected / reduced âœ“ |
| 4 | Model bakeoff report | evaluation/model_bakeoff_v1.md | Complete âœ“ |
| 5 | EDA / leakage summary | eda/leakage_risks.json | Reviewed âœ“ |
| 6 | Model card | model_cards/model_card_v1.md | Linked âœ“ |

> â˜• **BREAK â€” 10 minutes**



---
<div style="background: linear-gradient(135deg, #065F46 0%, #047857 100%); padding: 24px 32px; border-radius: 10px; color: white; margin: 16px 0;">
  <div style="font-size: 22px; font-weight: 900; margin-bottom: 4px;">ðŸš€ LAB 1.1 â€” PROMOTE, DEPLOY & OPERATIONALIZE</div>
  <div style="font-size: 14px; opacity: 0.85;">Steps 1â€“8 &nbsp;Â·&nbsp; ~45 minutes &nbsp;Â·&nbsp; Registry Â· Endpoint Â· EventBridge</div>
</div>

**Objective:** Take the registered candidate and turn it into a controlled production release with validation, approval, deployment, scheduling, and event-driven automation.



---
### ðŸ“¥ Step 1 â€” Pull the Candidate from the Registry
**â± ~3 min &nbsp;|&nbsp; Service: SageMaker Model Registry**

Always pull from the registry â€” never deploy from a local path. This enforces governance provenance.


In [21]:
run_venv(
    r"""
import sys
sys.path.insert(0, "/content")

import os
import io
import json
import boto3

def s3_download_json(s3_url):
    s3_path = s3_url.replace("s3://", "")
    bucket, key = s3_path.split("/", 1)
    s3 = boto3.client("s3")
    buf = io.BytesIO()
    s3.download_fileobj(bucket, key, buf)
    buf.seek(0)
    return json.load(buf)

# ── Configuration & S3 Path Setup ───────────────────────────────────────────
PROJECT = "model-engineering-lab"
REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
boto_session = boto3.session.Session(region_name=REGION)
sts_client = boto_session.client("sts")
ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

BUCKET = f"sagemaker-{REGION}-{ACCOUNT_ID}"
PREFIX = f"{PROJECT}"

PATHS = {
    "registry": f"s3://{BUCKET}/{PREFIX}/registry",
}

# ── Pull registry entry ─────────────────────────────────────────────────────
submission = s3_download_json(f"{PATHS['registry']}/candidate_submission_v1.json")

print("Registry submission:")
print(json.dumps({k: v for k, v in submission.items() if k != "model_package_arn"}, indent=2))
print(f"\nModel Package ARN : {submission['model_package_arn']}")
print(f"Approval Status   : {submission['approval_status']}")

# Confirm status is PendingManualApproval
assert submission["approval_status"] == "PendingManualApproval", (
    f"Expected PendingManualApproval, got {submission['approval_status']}"
)
print("\n✅ Gate passed: model is in PendingManualApproval state — ready for validation")

# ── Retrieve and display metrics from registry ───────────────────────────────
metrics_record = s3_download_json(submission["metrics_s3"])

print("\nMetrics from registry:")
for split in ["train", "val", "test"]:
    if split in metrics_record:
        m = metrics_record[split]
        print(f"  {split:>5}: AUC={m['auc']}  F1={m['f1']}  P={m['precision']}  R={m['recall']}")
"""
)

Registry submission:
{
  "model_package_group": "modelengineeringlab-churn",
  "sklearn_image_uri": "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3",
  "dataset_version": "v1",
  "split_version": "split_v1",
  "feature_config_id": "selected_v1",
  "training_commit_sha": "local-session-ad029fba",
  "metrics_s3": "s3://sagemaker-us-east-1-455865672536/model-engineering-lab/runs/run_002/metrics.json",
  "evaluation_s3": "s3://sagemaker-us-east-1-455865672536/model-engineering-lab/evaluation/model_bakeoff_v1.md",
  "leakage_test_s3": "s3://sagemaker-us-east-1-455865672536/model-engineering-lab/evaluation/shuffle_target_test.json",
  "approval_status": "PendingManualApproval",
  "created_at": "2026-09-19T13:33:37.998712Z"
}

Model Package ARN : arn:aws:sagemaker:us-east-1:455865672536:model-package/modelengineeringlab-churn/3
Approval Status   : PendingManualApproval

✅ Gate passed: model is in PendingManualApproval state — ready for validation

Metrics fr


---
### ðŸ”¬ Step 2 â€” Run Production Validation
**â± ~12 min &nbsp;|&nbsp; Service: S3**

Lightweight readiness check â€” not a repeat of Lab 1 experimentation. Focus is on runtime correctness.


In [22]:
run_venv(
    r"""
import sys
sys.path.insert(0, '/content')

import os
import io
import json
import tarfile
import joblib
import time as _time
import numpy as np
import pandas as pd
import boto3

from lab_helpers import s3_upload_json, s3_upload_text, s3_upload_df, now_iso

def s3_download_json(s3_url):
    s3_path = s3_url.replace('s3://', '')
    bucket, key = s3_path.split('/', 1)
    s3 = boto3.client('s3')
    buf = io.BytesIO()
    s3.download_fileobj(bucket, key, buf)
    buf.seek(0)
    return json.load(buf)

def s3_download_parquet(s3_url):
    s3_path = s3_url.replace('s3://', '')
    bucket, key = s3_path.split('/', 1)
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(io.BytesIO(obj['Body'].read()))

# ── Configuration & Dynamic S3 Path Setup ───────────────────────────────────
PROJECT = 'model-engineering-lab'
DATASET_VERSION = 'v1'
SPLIT_VERSION = 'split_v1'
PREPROCESS_VERSION = 'preprocess_v1'

REGION = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
boto_session = boto3.session.Session(region_name=REGION)
s3_client = boto_session.client('s3')
sts_client = boto_session.client('sts')
ACCOUNT_ID = sts_client.get_caller_identity()['Account']

BUCKET = f'sagemaker-{REGION}-{ACCOUNT_ID}'
PREFIX = f'{PROJECT}'

PATHS = {
    'raw':        f's3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/raw',
    'splits':     f's3://{BUCKET}/{PREFIX}/splits/{SPLIT_VERSION}',
    'runs':       f's3://{BUCKET}/{PREFIX}/runs',
    'evaluation': f's3://{BUCKET}/{PREFIX}/evaluation',
    'registry':   f's3://{BUCKET}/{PREFIX}/registry',
    'release':    f's3://{BUCKET}/{PREFIX}/release',
}

# ── Load Candidate & Validation Data ─────────────────────────────────────────
submission = s3_download_json(f"{PATHS['registry']}/candidate_submission_v1.json")
WINNER_RUN_ID = submission['metrics_s3'].split('/')[-2]
WINNER_FEAT_CFG = submission['feature_config_id']
leakage_test = s3_download_json(submission['leakage_test_s3'])

val_idx = s3_download_parquet(f"{PATHS['splits']}/valid_idx.parquet")['idx'].tolist()
df_full = s3_download_parquet(f"{PATHS['raw']}/bank_marketing_processed.parquet")
X_val = df_full.loc[val_idx].drop(columns=['target'])

# Load preprocessing pipeline from S3
pipe_bytes = io.BytesIO(
    s3_client.get_object(Bucket=BUCKET, Key=f'{PREFIX}/pipelines/{PREPROCESS_VERSION}.joblib')['Body'].read()
)
loaded_preprocessor = joblib.load(pipe_bytes)

# Dynamically parse selected features JSON structure
sel_data = s3_download_json(f's3://{BUCKET}/{PREFIX}/features/selected_features_v1.json')

if isinstance(sel_data, list):
    selected_idx = sel_data
elif isinstance(sel_data, dict):
    for possible_key in ['selected_indices', 'indices', 'selected_features', 'features']:
        if possible_key in sel_data:
            selected_idx = sel_data[possible_key]
            break
    else:
        selected_idx = next(v for v in sel_data.values() if isinstance(v, list))

# Dynamically transform validation data for base input
X_val_transformed = loaded_preprocessor.transform(X_val)
if WINNER_FEAT_CFG == 'selected_v1':
    if isinstance(selected_idx[0], str):
        X_val_base = loaded_preprocessor.transform(X_val[selected_idx])
    else:
        X_val_base = X_val_transformed[:, selected_idx]
elif WINNER_FEAT_CFG == 'reduced_v1':
    pca_bytes = io.BytesIO(s3_client.get_object(Bucket=BUCKET, Key=f'{PREFIX}/features/pca_transformer.joblib')['Body'].read())
    pca = joblib.load(pca_bytes)
    X_val_base = pca.transform(X_val_transformed)
else:
    X_val_base = X_val_transformed

# ── 1. Schema contract validation ───────────────────────────────────────────
inference_spec_loaded = s3_download_json(
    f"{PATHS['runs']}/{WINNER_RUN_ID}/inference_spec.json"
)

sample_payload = X_val_base[:5]
expected_input_shape = (None, inference_spec_loaded['input']['shape'][1])
actual_input_shape   = (None, sample_payload.shape[1])

schema_ok = expected_input_shape[1] == actual_input_shape[1]

schema_check = {
    'expected_input_features': inference_spec_loaded['input']['shape'][1],
    'actual_input_features':   sample_payload.shape[1],
    'schema_match':            schema_ok,
    'feature_config_id':       inference_spec_loaded['feature_config_id'],
    'checked_at':              now_iso()
}
print(f"Schema check: {'PASS' if schema_ok else 'FAIL'}")
print(json.dumps(schema_check, indent=2))

# ── 2. End-to-end smoke test ────────────────────────────────────────────────
model_bytes = io.BytesIO(
    s3_client.get_object(Bucket=BUCKET, Key=f"{PATHS['runs']}/{WINNER_RUN_ID}/model.tar.gz".replace(f"s3://{BUCKET}/", ""))['Body'].read()
)

try:
    with tarfile.open(fileobj=model_bytes, mode='r:gz') as tar:
        model_file = tar.extractfile('model.joblib')
        loaded_model = joblib.load(model_file)
except (tarfile.ReadError, Exception):
    model_bytes.seek(0)
    loaded_model = joblib.load(model_bytes)

smoke_raw = X_val.head(5)
smoke_transformed = loaded_preprocessor.transform(smoke_raw)

if WINNER_FEAT_CFG == 'selected_v1':
    if isinstance(selected_idx[0], str):
        smoke_input = loaded_preprocessor.transform(smoke_raw[selected_idx])
    else:
        smoke_input = smoke_transformed[:, selected_idx]
elif WINNER_FEAT_CFG == 'reduced_v1':
    smoke_input = pca.transform(smoke_transformed)
else:
    smoke_input = smoke_transformed

smoke_probs = loaded_model.predict_proba(smoke_input)[:, 1]

print("\nEnd-to-end smoke test passed")
print(f"   Sample predictions: {smoke_probs.round(4)}")
print(f"   All in [0,1]: {all(0 <= p <= 1 for p in smoke_probs)}")

# ── 3. Latency benchmark ────────────────────────────────────────────────────
latencies = []
for _ in range(100):
    row = X_val.sample(1)
    t0 = _time.perf_counter()
    if WINNER_FEAT_CFG == 'selected_v1' and isinstance(selected_idx[0], str):
        t = loaded_preprocessor.transform(row[selected_idx])
    else:
        t = loaded_preprocessor.transform(row)
        if WINNER_FEAT_CFG == 'selected_v1':
            t = t[:, selected_idx]
        elif WINNER_FEAT_CFG == 'reduced_v1':
            t = pca.transform(t)
    loaded_model.predict_proba(t)
    latencies.append((_time.perf_counter() - t0) * 1000)

latency_stats = {
    'p50_ms': round(float(np.percentile(latencies, 50)), 3),
    'p95_ms': round(float(np.percentile(latencies, 95)), 3),
    'p99_ms': round(float(np.percentile(latencies, 99)), 3),
    'mean_ms': round(float(np.mean(latencies)), 3),
    'samples': 100,
    'within_sla_10ms_pct': round(float(np.mean([l < 10 for l in latencies]) * 100), 1)
}

print("\nLatency benchmark (100 single-row predictions):")
print(json.dumps(latency_stats, indent=2))

# ── 4. Batch scoring smoke test ─────────────────────────────────────────────
SMOKE_BATCH_SIZE = 500
smoke_batch_raw = X_val.head(SMOKE_BATCH_SIZE)

if WINNER_FEAT_CFG == 'selected_v1' and isinstance(selected_idx[0], str):
    smoke_batch_input = loaded_preprocessor.transform(smoke_batch_raw[selected_idx])
else:
    smoke_batch_t = loaded_preprocessor.transform(smoke_batch_raw)
    if WINNER_FEAT_CFG == 'selected_v1':
        smoke_batch_input = smoke_batch_t[:, selected_idx]
    elif WINNER_FEAT_CFG == 'reduced_v1':
        smoke_batch_input = pca.transform(smoke_batch_t)
    else:
        smoke_batch_input = smoke_batch_t

smoke_batch_probs = loaded_model.predict_proba(smoke_batch_input)[:, 1]
smoke_batch_preds = (smoke_batch_probs > 0.5).astype(int)

smoke_df = smoke_batch_raw.copy()
smoke_df['score']      = smoke_batch_probs
smoke_df['prediction'] = smoke_batch_preds

print(f"\nBatch smoke test: {SMOKE_BATCH_SIZE} rows scored without error")
print(f"   Predicted positive rate: {smoke_batch_preds.mean():.3f}")
print(f"   Score range: [{smoke_batch_probs.min():.4f}, {smoke_batch_probs.max():.4f}]")

# ── Persist validation artifacts ────────────────────────────────────────────
s3_upload_json(schema_check, f"{PATHS['release']}/schema_check.json")
s3_upload_json(latency_stats, f"{PATHS['release']}/latency_smoke.json")
s3_upload_df(smoke_df, f"{PATHS['release']}/batch_smoke_predictions.parquet")

report_lines = [
    '# Release Validation Report — v1',
    f'Generated: {now_iso()}',
    '',
    '## Schema Check',
    f"- Expected features: {schema_check['expected_input_features']}",
    f"- Actual features: {schema_check['actual_input_features']}",
    f"- Result: {'PASS' if schema_check['schema_match'] else 'FAIL'}",
    '',
    '## Smoke Test',
    '- End-to-end (preprocessing + model): PASS',
    f'- Batch ({SMOKE_BATCH_SIZE} rows): PASS',
    '',
    '## Latency',
    f"- P50: {latency_stats['p50_ms']} ms",
    f"- P95: {latency_stats['p95_ms']} ms",
    f"- P99: {latency_stats['p99_ms']} ms",
    f"- Within 10ms SLA: {latency_stats['within_sla_10ms_pct']}%",
    '',
    '## Leakage',
    f"- Shuffle-target test: {'PASS' if leakage_test['passed'] else 'WARNING'}",
    '',
    '## Recommendation',
    'APPROVE — all checks passed' if schema_check['schema_match'] and leakage_test['passed'] else 'INVESTIGATE — one or more checks flagged'
]

validation_report = '\n'.join(report_lines)
s3_upload_text(validation_report, f"{PATHS['release']}/validation_report_v1.md")
print("\nValidation artifacts uploaded successfully")
"""
)

Schema check: PASS
{
  "expected_input_features": 20,
  "actual_input_features": 20,
  "schema_match": true,
  "feature_config_id": "selected_v1",
  "checked_at": "2026-09-19T13:34:07.093729Z"
}

End-to-end smoke test passed
   Sample predictions: [0.029  0.2894 0.0781 0.0331 0.0593]
   All in [0,1]: True

Latency benchmark (100 single-row predictions):
{
  "p50_ms": 48.868,
  "p95_ms": 51.369,
  "p99_ms": 58.754,
  "mean_ms": 49.173,
  "samples": 100,
  "within_sla_10ms_pct": 0.0
}

Batch smoke test: 500 rows scored without error
   Predicted positive rate: 0.004
   Score range: [0.0290, 0.5309]
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/release/schema_check.json
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/release/latency_smoke.json
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/release/batch_smoke_predictions.parquet
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab


---
### âœ… Step 3 â€” Approve or Reject
**â± ~3 min &nbsp;|&nbsp; Service: SageMaker Model Registry**

Formal gate. This is a deliberate human action â€” not automated. In regulated industries a second reviewer may be required.


In [23]:
run_venv(
    r"""
import sys
sys.path.insert(0, '/content')

import os
import io
import json
import boto3

from lab_helpers import s3_upload_json, now_iso

def s3_download_json(s3_url):
    s3_path = s3_url.replace('s3://', '')
    bucket, key = s3_path.split('/', 1)
    s3 = boto3.client('s3')
    buf = io.BytesIO()
    s3.download_fileobj(bucket, key, buf)
    buf.seek(0)
    return json.load(buf)

# ── Configuration & Path Setup ───────────────────────────────────────────────
PROJECT = 'model-engineering-lab'
REGION = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
boto_session = boto3.session.Session(region_name=REGION)
sm_client = boto_session.client('sagemaker')
sts_client = boto_session.client('sts')
ACCOUNT_ID = sts_client.get_caller_identity()['Account']

BUCKET = f'sagemaker-{REGION}-{ACCOUNT_ID}'
PREFIX = f'{PROJECT}'

PATHS = {
    'registry': f's3://{BUCKET}/{PREFIX}/registry',
    'release':  f's3://{BUCKET}/{PREFIX}/release',
}

# ── Load Validation Artifacts ────────────────────────────────────────────────
submission = s3_download_json(f"{PATHS['registry']}/candidate_submission_v1.json")
schema_check = s3_download_json(f"{PATHS['release']}/schema_check.json")
latency_stats = s3_download_json(f"{PATHS['release']}/latency_smoke.json")
leakage_test = s3_download_json(submission['leakage_test_s3'])
winner_metrics = s3_download_json(submission['metrics_s3'])

MODEL_PACKAGE_ARN = submission.get('model_package_arn')

# ── Run Approval Gate Logic ──────────────────────────────────────────────────
schema_passed   = schema_check["schema_match"]
leakage_passed  = leakage_test["passed"]
metrics_passed  = winner_metrics["val"]["auc"] >= 0.75
latency_passed  = latency_stats["p95_ms"] < 50

gate_results = {
    "schema_check":      schema_passed,
    "leakage_check":     leakage_passed,
    "metrics_threshold": metrics_passed,
    "latency_sla":       latency_passed,
}

all_passed = all(gate_results.values())
APPROVAL_STATUS = "Approved" if all_passed else "Rejected"

print("Gate Results:")
for gate, result in gate_results.items():
    print(f"   {'✅' if result else '❌'} {gate}: {'PASS' if result else 'FAIL'}")

print(f"\n{'✅' if all_passed else '❌'} Decision: {APPROVAL_STATUS}")

# ── Update Registry Approval Status ──────────────────────────────────────────
if MODEL_PACKAGE_ARN:
    try:
        sm_client.update_model_package(
            ModelPackageArn=MODEL_PACKAGE_ARN,
            ModelApprovalStatus=APPROVAL_STATUS,
            ApprovalDescription=(
                f"Auto-gate passed: {gate_results}. Reviewer: lab-session. Time: {now_iso()}"
                if all_passed
                else f"Gate failure: {gate_results}"
            )
        )
        print(f"✅ Registry status updated to: {APPROVAL_STATUS}")
    except Exception as e:
        print(f"⚠️  Registry update: {e}  (continuing — status set locally)")
else:
    print("ℹ️  No ModelPackageArn found in submission; skipping SageMaker registry update.")

# ── Update Local Submission Record ───────────────────────────────────────────
submission["approval_status"] = APPROVAL_STATUS
submission["approval_gate_results"] = gate_results
submission["approved_at"] = now_iso()
s3_upload_json(submission, f"{PATHS['registry']}/candidate_submission_v1.json")
print("✅ Candidate submission record updated in S3")
"""
)

Gate Results:
   ✅ schema_check: PASS
   ✅ leakage_check: PASS
   ✅ metrics_threshold: PASS
   ❌ latency_sla: FAIL

❌ Decision: Rejected
✅ Registry status updated to: Rejected
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/registry/candidate_submission_v1.json
✅ Candidate submission record updated in S3



---
### ðŸ“¦ Step 4 â€” Package Deployment Assets
**â± ~5 min &nbsp;|&nbsp; Service: S3**

All assets must be bundled and versioned together. A partial bundle is not deployable.


In [24]:
run_venv(
    r"""
import sys
sys.path.insert(0, '/content')

import os
import io
import json
import tarfile
import joblib
import tracemalloc
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
import boto3

from lab_helpers import s3_upload_json, s3_upload_text, s3_upload_bytes, now_iso

def s3_download_json(s3_url):
    s3_path = s3_url.replace('s3://', '')
    bucket, key = s3_path.split('/', 1)
    s3 = boto3.client('s3')
    buf = io.BytesIO()
    s3.download_fileobj(bucket, key, buf)
    buf.seek(0)
    return json.load(buf)

def s3_download_parquet(s3_url):
    s3_path = s3_url.replace('s3://', '')
    bucket, key = s3_path.split('/', 1)
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(io.BytesIO(obj['Body'].read()))

# ── Setup & Paths ────────────────────────────────────────────────────────────
PROJECT = 'model-engineering-lab'
DATASET_VERSION = 'v1'
SPLIT_VERSION = 'split_v1'
PREPROCESS_VERSION = 'preprocess_v1'
DEPLOYMENT_INSTANCE_TYPE = 'ml.m5.large'

REGION = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
boto_session = boto3.session.Session(region_name=REGION)
s3_client = boto_session.client('s3')
sts_client = boto_session.client('sts')
ACCOUNT_ID = sts_client.get_caller_identity()['Account']

BUCKET = f'sagemaker-{REGION}-{ACCOUNT_ID}'
PREFIX = f'{PROJECT}'

PATHS = {
    'raw':        f's3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/raw',
    'splits':     f's3://{BUCKET}/{PREFIX}/splits/{SPLIT_VERSION}',
    'runs':       f's3://{BUCKET}/{PREFIX}/runs',
    'registry':   f's3://{BUCKET}/{PREFIX}/registry',
    'release':    f's3://{BUCKET}/{PREFIX}/release',
    'deploy':     f's3://{BUCKET}/{PREFIX}/deploy',
}

# ── 1. Write inference.py entry point ────────────────────────────────────────
inference_py = '''import os
import io
import json
import joblib
import numpy as np


def _load_optional(model_dir, filename):
    path = os.path.join(model_dir, filename)
    return joblib.load(path) if os.path.exists(path) else None


def model_fn(model_dir):
    model = joblib.load(os.path.join(model_dir, "model.joblib"))
    preprocessor = joblib.load(os.path.join(model_dir, "preprocessor.joblib"))
    selected_idx = _load_optional(model_dir, "selected_idx.joblib")
    pca = _load_optional(model_dir, "pca.joblib")

    cfg_path = os.path.join(model_dir, "feature_config_id.txt")
    feature_config_id = "baseline_v1"
    if os.path.exists(cfg_path):
        with open(cfg_path) as f:
            feature_config_id = f.read().strip()

    return {
        "model": model,
        "preprocessor": preprocessor,
        "selected_idx": selected_idx,
        "pca": pca,
        "feature_config_id": feature_config_id,
    }


def _to_text(request_body):
    if isinstance(request_body, (bytes, bytearray)):
        return request_body.decode("utf-8")
    return request_body


def input_fn(request_body, content_type="application/json"):
    import pandas as pd

    request_body = _to_text(request_body)
    content_type = (content_type or "application/json").split(";")[0].strip().lower()

    if content_type == "application/json":
        data = json.loads(request_body)
        if isinstance(data, dict) and "instances" in data:
            data = data["instances"]
        return pd.DataFrame(data if isinstance(data, list) else [data])

    if content_type == "text/csv":
        return pd.read_csv(io.StringIO(request_body))

    raise ValueError(f"Unsupported content type: {content_type}")


def predict_fn(input_data, model_artifacts):
    X = model_artifacts["preprocessor"].transform(input_data)
    cfg = model_artifacts.get("feature_config_id", "baseline_v1")

    if cfg == "selected_v1":
        selected_idx = model_artifacts.get("selected_idx")
        if selected_idx is None:
            raise ValueError("selected_v1 requires selected_idx.joblib in the model bundle")
        X = X[:, selected_idx]

    elif cfg == "reduced_v1":
        pca = model_artifacts.get("pca")
        if pca is None:
            raise ValueError("reduced_v1 requires pca.joblib in the model bundle")
        X = pca.transform(X)

    proba = model_artifacts["model"].predict_proba(X)[:, 1]
    return np.asarray(proba, dtype=float)


def output_fn(prediction, accept="application/json"):
    accept = (accept or "application/json").split(";")[0].strip().lower()
    body = json.dumps({"predictions": prediction.tolist()})

    if accept in ("application/json", "*/*"):
        return body, "application/json"

    return body, "application/json"
'''

s3_upload_text(inference_py, f"{PATHS['deploy']}/inference.py")
print("✅ inference.py uploaded")

# ── Load Model & Artifacts for Checks ────────────────────────────────────────
submission = s3_download_json(f"{PATHS['registry']}/candidate_submission_v1.json")
WINNER_RUN_ID = submission['metrics_s3'].split('/')[-2]
WINNER_FEAT_CFG = submission['feature_config_id']
MODEL_PACKAGE_ARN = submission.get('model_package_arn', 'N/A')

val_idx = s3_download_parquet(f"{PATHS['splits']}/valid_idx.parquet")['idx'].tolist()
df_full = s3_download_parquet(f"{PATHS['raw']}/bank_marketing_processed.parquet")
X_val = df_full.loc[val_idx].drop(columns=['target'])
y_val = df_full.loc[val_idx]['target']

pipe_bytes = io.BytesIO(s3_client.get_object(Bucket=BUCKET, Key=f'{PREFIX}/pipelines/{PREPROCESS_VERSION}.joblib')['Body'].read())
loaded_preprocessor = joblib.load(pipe_bytes)

model_bytes = io.BytesIO(s3_client.get_object(Bucket=BUCKET, Key=f"{PATHS['runs']}/{WINNER_RUN_ID}/model.tar.gz".replace(f"s3://{BUCKET}/", ""))['Body'].read())
try:
    with tarfile.open(fileobj=model_bytes, mode='r:gz') as tar:
        loaded_model = joblib.load(tar.extractfile('model.joblib'))
except Exception:
    model_bytes.seek(0)
    loaded_model = joblib.load(model_bytes)

sel_data = s3_download_json(f's3://{BUCKET}/{PREFIX}/features/selected_features_v1.json')
if isinstance(sel_data, list):
    selected_idx = sel_data
elif isinstance(sel_data, dict):
    for k in ['selected_indices', 'indices', 'selected_features', 'features']:
        if k in sel_data:
            selected_idx = sel_data[k]
            break
    else:
        selected_idx = next(v for v in sel_data.values() if isinstance(v, list))

# Get feature output metadata
if hasattr(loaded_preprocessor, "get_feature_names_out"):
    feature_names_out = list(loaded_preprocessor.get_feature_names_out())
else:
    feature_names_out = [f"feature_{i}" for i in range(loaded_preprocessor.transform(X_val.head(1)).shape[1])]

if isinstance(selected_idx[0], str):
    final_selected = selected_idx
else:
    final_selected = [feature_names_out[i] for i in selected_idx]

pca = None
if WINNER_FEAT_CFG == "reduced_v1":
    pca_bytes = io.BytesIO(s3_client.get_object(Bucket=BUCKET, Key=f'{PREFIX}/features/pca_transformer.joblib')['Body'].read())
    pca = joblib.load(pca_bytes)

# ── 5. Threshold sanity check (classification) ──────────────────────────────
X_val_prepared = loaded_preprocessor.transform(X_val)

if WINNER_FEAT_CFG == "baseline_v1":
    X_val_model = X_val_prepared
elif WINNER_FEAT_CFG == "reduced_v1":
    X_val_model = pca.transform(X_val_prepared)
elif WINNER_FEAT_CFG == "selected_v1":
    if isinstance(selected_idx[0], str):
        X_val_model = loaded_preprocessor.transform(X_val[selected_idx])
    else:
        X_val_model = X_val_prepared[:, selected_idx]
else:
    raise ValueError(f"Unknown WINNER_FEAT_CFG: {WINNER_FEAT_CFG}")

probs_val = loaded_model.predict_proba(X_val_model)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, probs_val)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, precisions[:-1], label="Precision", color="#1D4ED8")
ax.plot(thresholds, recalls[:-1],    label="Recall",    color="#065F46")
ax.axvline(0.5, linestyle="--", color="gray", label="Default threshold=0.5")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision-Recall vs Threshold", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("precision_recall_threshold.png")
plt.close()

f1_at_thresholds = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
best_idx = f1_at_thresholds.argmax()
best_thresh = thresholds[best_idx]

print(f"Recommended threshold (max F1): {best_thresh:.3f}")
print(f"At threshold {best_thresh:.3f}: Precision={precisions[best_idx]:.3f}  Recall={recalls[best_idx]:.3f}")

# ── 6. Memory footprint sanity ──────────────────────────────────────────────
tracemalloc.start()
_ = loaded_preprocessor.transform(X_val.head(1000))
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"\nMemory footprint (1000-row transform): peak={peak/1024:.1f} KB")

# ── 7. Rollback compatibility check ─────────────────────────────────────────
print("\nRollback compatibility:")
print("  Previous approved version ARN: (set in deploy_config.yaml -> rollback_model_arn)")
print("  Compatibility requirement: input schema must not change between versions")
print(f"  Current input features: {len(feature_names_out)}")
print("  ✅ Rollback path is documented — update rollback_model_arn before production promotion")

# ── Write input/output schemas & deploy config ────────────────────────────────
N_COMPONENTS = pca.n_components_ if pca is not None else 10

input_schema = {
    "content_type": ["application/json", "text/csv"],
    "feature_config_id": WINNER_FEAT_CFG,
    "expected_features": (
        feature_names_out if WINNER_FEAT_CFG == "baseline_v1"
        else final_selected if WINNER_FEAT_CFG == "selected_v1"
        else [f"pca_{i}" for i in range(N_COMPONENTS)]
    ),
    "example": {"age": 35, "job": "admin.", "balance": 1500, "duration": 180}
}

output_schema = {
    "content_type": "application/json",
    "response_field": "predictions",
    "value_type": "float32",
    "range": [0.0, 1.0],
    "interpretation": "probability of term deposit subscription"
}

s3_upload_json(input_schema,  f"{PATHS['deploy']}/input_schema.json")
s3_upload_json(output_schema, f"{PATHS['deploy']}/output_schema.json")

deploy_config = {
    "instance_type": DEPLOYMENT_INSTANCE_TYPE,
    "initial_instance_count": 1,
    "model_package_arn": MODEL_PACKAGE_ARN,
    "endpoint_name": f"{PROJECT}-endpoint",
    "data_capture": {"enable": True, "sampling_pct": 20},
    "auto_scaling": {
        "min_capacity": 1,
        "max_capacity": 3,
        "target_invocations_per_instance": 100
    },
    "rollback_model_arn": "PREVIOUS_APPROVED_VERSION_ARN_HERE"
}
s3_upload_bytes(yaml.dump(deploy_config).encode(), f"{PATHS['deploy']}/deploy_config.yaml")
print("✅ All deployment assets packaged and uploaded")
"""
)

   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/deploy/inference.py
✅ inference.py uploaded
Recommended threshold (max F1): 0.142
At threshold 0.142: Precision=0.191  Recall=0.724

Memory footprint (1000-row transform): peak=430.3 KB

Rollback compatibility:
  Previous approved version ARN: (set in deploy_config.yaml -> rollback_model_arn)
  Compatibility requirement: input schema must not change between versions
  Current input features: 20
  ✅ Rollback path is documented — update rollback_model_arn before production promotion
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/deploy/input_schema.json
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/deploy/output_schema.json
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/deploy/deploy_config.yaml
✅ All deployment assets packaged and uploaded


---
### ðŸŒ Step 5 â€” Deploy

The deployment cells below use boto3 only. They intentionally package model artifacts and `inference.py` as **separate S3 tarballs**, matching SageMaker script-mode behavior without importing the SageMaker Python SDK. This avoids the repeated `No module named inference` endpoint failure caused by placing code directly inside the model artifact tarball.


In [25]:
run_venv(
    r"""
import sys
sys.path.insert(0, '/content')

import os
import io
import json
import time
import tarfile
import tempfile
import joblib
import yaml
import numpy as np
import pandas as pd
import boto3
import sagemaker
from urllib.parse import urlparse

from lab_helpers import s3_upload_json, s3_upload_text, s3_upload_bytes, now_iso

def s3_download_json(s3_url):
    s3_path = s3_url.replace('s3://', '')
    bucket, key = s3_path.split('/', 1)
    s3 = boto3.client('s3')
    buf = io.BytesIO()
    s3.download_fileobj(bucket, key, buf)
    buf.seek(0)
    return json.load(buf)

def s3_download_parquet(s3_url):
    s3_path = s3_url.replace('s3://', '')
    bucket, key = s3_path.split('/', 1)
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(io.BytesIO(obj['Body'].read()))

# ── 1. Load context directly from S3 ─────────────────────────────────────────
PROJECT = 'model-engineering-lab'
DATASET_VERSION = 'v1'
SPLIT_VERSION = 'split_v1'
PREPROCESS_VERSION = 'preprocess_v1'
REGION = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')

session = boto3.Session(region_name=REGION)
sm_client = session.client('sagemaker')
s3_client = session.client('s3')
runtime_client = session.client('sagemaker-runtime')
sts_client = session.client('sts')

ACCOUNT_ID = sts_client.get_caller_identity()['Account']
BUCKET = f'sagemaker-{REGION}-{ACCOUNT_ID}'
PREFIX = f'{PROJECT}'

PATHS = {
    'raw':        f's3://{BUCKET}/{PREFIX}/datasets/{DATASET_VERSION}/raw',
    'splits':     f's3://{BUCKET}/{PREFIX}/splits/{SPLIT_VERSION}',
    'runs':       f's3://{BUCKET}/{PREFIX}/runs',
    'registry':   f's3://{BUCKET}/{PREFIX}/registry',
    'release':    f's3://{BUCKET}/{PREFIX}/release',
    'deploy':     f's3://{BUCKET}/{PREFIX}/deploy',
}

submission = s3_download_json(f"{PATHS['registry']}/candidate_submission_v1.json")
WINNER_RUN_ID = submission['metrics_s3'].split('/')[-2]
WINNER_FEAT_CFG = submission['feature_config_id']

val_idx = s3_download_parquet(f"{PATHS['splits']}/valid_idx.parquet")['idx'].tolist()
df_full = s3_download_parquet(f"{PATHS['raw']}/bank_marketing_processed.parquet")
X_val = df_full.loc[val_idx].drop(columns=['target'])

pipe_bytes = io.BytesIO(s3_client.get_object(Bucket=BUCKET, Key=f'{PREFIX}/pipelines/{PREPROCESS_VERSION}.joblib')['Body'].read())
loaded_preprocessor = joblib.load(pipe_bytes)

model_bytes = io.BytesIO(s3_client.get_object(Bucket=BUCKET, Key=f"{PATHS['runs']}/{WINNER_RUN_ID}/model.tar.gz".replace(f"s3://{BUCKET}/", ""))['Body'].read())
try:
    with tarfile.open(fileobj=model_bytes, mode='r:gz') as tar:
        loaded_model = joblib.load(tar.extractfile('model.joblib'))
except Exception:
    model_bytes.seek(0)
    loaded_model = joblib.load(model_bytes)

selected_idx = None
if WINNER_FEAT_CFG == 'selected_v1':
    sel_data = s3_download_json(f's3://{BUCKET}/{PREFIX}/features/selected_features_v1.json')
    if isinstance(sel_data, list):
        selected_idx = sel_data
    elif isinstance(sel_data, dict):
        selected_idx = next(v for v in sel_data.values() if isinstance(v, list))

pca = None
if WINNER_FEAT_CFG == "reduced_v1":
    pca_bytes = io.BytesIO(s3_client.get_object(Bucket=BUCKET, Key=f'{PREFIX}/features/pca_transformer.joblib')['Body'].read())
    pca = joblib.load(pca_bytes)

# ── 2. Define inference handler ─────────────────────────────────────────────
inference_py = '''import os
import io
import json
import joblib
import numpy as np

def _load_optional(model_dir, filename):
    path = os.path.join(model_dir, filename)
    return joblib.load(path) if os.path.exists(path) else None

def model_fn(model_dir):
    model = joblib.load(os.path.join(model_dir, "model.joblib"))
    preprocessor = joblib.load(os.path.join(model_dir, "preprocessor.joblib"))
    selected_idx = _load_optional(model_dir, "selected_idx.joblib")
    pca = _load_optional(model_dir, "pca.joblib")

    cfg_path = os.path.join(model_dir, "feature_config_id.txt")
    feature_config_id = "baseline_v1"
    if os.path.exists(cfg_path):
        with open(cfg_path) as f:
            feature_config_id = f.read().strip()

    return {
        "model": model,
        "preprocessor": preprocessor,
        "selected_idx": selected_idx,
        "pca": pca,
        "feature_config_id": feature_config_id,
    }

def _to_text(request_body):
    if isinstance(request_body, (bytes, bytearray)):
        return request_body.decode("utf-8")
    return request_body

def input_fn(request_body, content_type="application/json"):
    import pandas as pd
    request_body = _to_text(request_body)
    content_type = (content_type or "application/json").split(";")[0].strip().lower()

    if content_type == "application/json":
        data = json.loads(request_body)
        if isinstance(data, dict) and "instances" in data:
            data = data["instances"]
        return pd.DataFrame(data if isinstance(data, list) else [data])

    if content_type == "text/csv":
        return pd.read_csv(io.StringIO(request_body))

    raise ValueError(f"Unsupported content type: {content_type}")

def predict_fn(input_data, model_artifacts):
    preprocessor = model_artifacts["preprocessor"]
    X = preprocessor.transform(input_data)

    # Ensure X is a dense NumPy array
    if hasattr(X, "toarray"):
        X = X.toarray()
    elif hasattr(X, "to_numpy"):
        X = X.to_numpy()
    else:
        X = np.asarray(X)

    cfg = model_artifacts.get("feature_config_id", "baseline_v1")

    if cfg == "selected_v1":
        selected_idx = model_artifacts.get("selected_idx")
        if selected_idx is None:
            raise ValueError("selected_v1 requires selected_idx.joblib in the model bundle")

        # Handle case where selected_idx contains column names instead of integer positions
        if len(selected_idx) > 0 and isinstance(selected_idx[0], str):
            if hasattr(preprocessor, "get_feature_names_out"):
                feature_names = list(preprocessor.get_feature_names_out())
            else:
                feature_names = list(input_data.columns)

            int_idx = []
            for item in selected_idx:
                if item in feature_names:
                    int_idx.append(feature_names.index(item))
                elif item in input_data.columns:
                    int_idx.append(list(input_data.columns).index(item))
                else:
                    try:
                        int_idx.append(int(item))
                    except ValueError:
                        raise ValueError(f"Feature name '{item}' not found in preprocessor output feature names.")
        else:
            int_idx = [int(i) for i in selected_idx]

        X = X[:, int_idx]

    elif cfg == "reduced_v1":
        pca = model_artifacts.get("pca")
        if pca is None:
            raise ValueError("reduced_v1 requires pca.joblib in the model bundle")
        X = pca.transform(X)

    proba = model_artifacts["model"].predict_proba(X)[:, 1]
    return np.asarray(proba, dtype=float)

def output_fn(prediction, accept="application/json"):
    accept = (accept or "application/json").split(";")[0].strip().lower()
    body = json.dumps({"predictions": prediction.tolist()})
    return body, "application/json"
'''

# ── 3. Local smoke test ──────────────────────────────────────────────────────
with tempfile.TemporaryDirectory() as smoke_dir:
    joblib.dump(loaded_model, os.path.join(smoke_dir, "model.joblib"))
    joblib.dump(loaded_preprocessor, os.path.join(smoke_dir, "preprocessor.joblib"))
    with open(os.path.join(smoke_dir, "feature_config_id.txt"), "w") as f:
        f.write(WINNER_FEAT_CFG)
    if WINNER_FEAT_CFG == "selected_v1":
        joblib.dump(selected_idx, os.path.join(smoke_dir, "selected_idx.joblib"))
    elif WINNER_FEAT_CFG == "reduced_v1":
        joblib.dump(pca, os.path.join(smoke_dir, "pca.joblib"))

    namespace = {}
    exec(inference_py, namespace)
    artifacts = namespace["model_fn"](smoke_dir)
    sample_payload = X_val.head(3).to_json(orient="records")
    sample_input = namespace["input_fn"](sample_payload, "application/json")
    sample_pred = namespace["predict_fn"](sample_input, artifacts)
    sample_output = namespace["output_fn"](sample_pred, "application/json")

print("✅ Local inference.py smoke test passed")
print("Sample output:", sample_output)

# ── 4. Package & Upload Model and Source Archives ───────────────────────────
package_ts = int(time.time())
endpoint_model_filename = f"endpoint_model_artifacts_{package_ts}.tar.gz"
endpoint_source_filename = f"endpoint_source_{package_ts}.tar.gz"

endpoint_model_path = f"/tmp/{endpoint_model_filename}"
endpoint_source_path = f"/tmp/{endpoint_source_filename}"
endpoint_model_artifact_s3 = f"{PATHS['deploy']}/{endpoint_model_filename}"
endpoint_source_s3 = f"{PATHS['deploy']}/{endpoint_source_filename}"

with tempfile.TemporaryDirectory() as tmpdir:
    joblib.dump(loaded_model, os.path.join(tmpdir, "model.joblib"))
    joblib.dump(loaded_preprocessor, os.path.join(tmpdir, "preprocessor.joblib"))
    with open(os.path.join(tmpdir, "feature_config_id.txt"), "w") as f:
        f.write(WINNER_FEAT_CFG)
    if WINNER_FEAT_CFG == "selected_v1":
        joblib.dump(selected_idx, os.path.join(tmpdir, "selected_idx.joblib"))
    elif WINNER_FEAT_CFG == "reduced_v1":
        joblib.dump(pca, os.path.join(tmpdir, "pca.joblib"))

    with tarfile.open(endpoint_model_path, "w:gz") as tar:
        tar.add(os.path.join(tmpdir, "model.joblib"), arcname="model.joblib")
        tar.add(os.path.join(tmpdir, "preprocessor.joblib"), arcname="preprocessor.joblib")
        tar.add(os.path.join(tmpdir, "feature_config_id.txt"), arcname="feature_config_id.txt")
        if WINNER_FEAT_CFG == "selected_v1":
            tar.add(os.path.join(tmpdir, "selected_idx.joblib"), arcname="selected_idx.joblib")
        elif WINNER_FEAT_CFG == "reduced_v1":
            tar.add(os.path.join(tmpdir, "pca.joblib"), arcname="pca.joblib")

with tempfile.TemporaryDirectory() as srcdir:
    with open(os.path.join(srcdir, "inference.py"), "w") as f:
        f.write(inference_py)
    with tarfile.open(endpoint_source_path, "w:gz") as tar:
        tar.add(os.path.join(srcdir, "inference.py"), arcname="inference.py")

with open(endpoint_model_path, "rb") as f:
    s3_upload_bytes(f.read(), endpoint_model_artifact_s3)
with open(endpoint_source_path, "rb") as f:
    s3_upload_bytes(f.read(), endpoint_source_s3)

print("✅ Uploaded Model Archive:", endpoint_model_artifact_s3)
print("✅ Uploaded Source Archive:", endpoint_source_s3)

# ── 5. Create & Deploy SageMaker Endpoint ────────────────────────────────────
try:
    ROLE = sagemaker.get_execution_role(sagemaker_session=sagemaker.Session(boto_session=session))
except Exception:
    ROLE = f"arn:aws:iam::{ACCOUNT_ID}:role/model-engineering-lab-sagemaker-execution"

try:
    sklearn_image_uri = sagemaker.image_uris.retrieve(
        framework="scikit-learn", region=REGION, version="1.2-1", py_version="py3", instance_type="ml.m5.large"
    )
except Exception:
    sklearn_image_uri = f"683313688378.dkr.ecr.{REGION}.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

deploy_ts = int(time.time())
MODEL_NAME = f"{PROJECT}-model-{deploy_ts}"
ENDPOINT_CONFIG_NAME = f"{PROJECT}-epc-{deploy_ts}"
ENDPOINT_NAME = f"{PROJECT}-ep-{deploy_ts}"

sm_client.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=ROLE,
    PrimaryContainer={
        "Image": sklearn_image_uri,
        "ModelDataUrl": endpoint_model_artifact_s3,
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": endpoint_source_s3,
            "SAGEMAKER_CONTAINER_LOG_LEVEL": "20",
            "SAGEMAKER_REGION": REGION,
        },
    },
)

sm_client.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": MODEL_NAME,
        "ServerlessConfig": {"MemorySizeInMB": 2048, "MaxConcurrency": 2},
    }],
)

sm_client.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_CONFIG_NAME)
print("🚀 Endpoint creation initiated:", ENDPOINT_NAME)

while True:
    desc = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    status = desc["EndpointStatus"]
    if status == "InService":
        print("✅ Endpoint is live and ready.")
        break
    elif status == "Failed":
        raise RuntimeError(f"Deployment failed: {desc.get('FailureReason')}")
    time.sleep(15)

# ── 6. Live Endpoint Invocation ───────────────────────────────────────────────
test_payload = X_val.head(3).to_dict(orient="records")
response = runtime_client.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps(test_payload),
)

result = response["Body"].read().decode("utf-8")
print("✅ Live endpoint test result:")
print(result)
"""
)

✅ Local inference.py smoke test passed
Sample output: ('{"predictions": [0.010098416073353655, 0.036064324903321315, 0.015525489858095133]}', 'application/json')
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/deploy/endpoint_model_artifacts_1789824894.tar.gz
   ✅ uploaded → s3://sagemaker-us-east-1-455865672536/model-engineering-lab/deploy/endpoint_source_1789824894.tar.gz
✅ Uploaded Model Archive: s3://sagemaker-us-east-1-455865672536/model-engineering-lab/deploy/endpoint_model_artifacts_1789824894.tar.gz
✅ Uploaded Source Archive: s3://sagemaker-us-east-1-455865672536/model-engineering-lab/deploy/endpoint_source_1789824894.tar.gz
🚀 Endpoint creation initiated: model-engineering-lab-ep-1789824901
✅ Endpoint is live and ready.
✅ Live endpoint test result:
{"predictions": [0.010098416073353654, 0.036064324903321315, 0.015525489858095133]}



---
### â° Step 6 â€” Schedule Retraining or Scoring with EventBridge Scheduler
**â± ~5 min &nbsp;|&nbsp; Service: EventBridge Scheduler**

Use EventBridge Scheduler for cron-based ML operations. Supports one-time, rate-based, and cron schedules.

> ðŸ’¡ **Teaching Point:** Retry policy is not optional. A failed nightly scoring job with no retry means the downstream application receives stale predictions â€” potentially for 24 hours.


In [26]:
run_venv(
    r"""
import os
import boto3

# ── Setup AWS Context ────────────────────────────────────────────────────────
PROJECT = 'model-engineering-lab'
REGION = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')

session = boto3.Session(region_name=REGION)
sts_client = session.client('sts')
eb_scheduler = session.client('scheduler')

ACCOUNT_ID = sts_client.get_caller_identity()['Account']

# ── Get or create EventBridge Scheduler role ───────────────────────────────────
schedules = [
    {
        "name":        f"{PROJECT}-nightly-batch-scoring",
        "schedule":    "cron(0 2 * * ? *)",      # 2am UTC every day
        "description": "Nightly batch scoring of customer base",
        "target_arn":  f"arn:aws:sagemaker:{REGION}:{ACCOUNT_ID}:transform-job/*",
    },
    {
        "name":        f"{PROJECT}-weekly-retrain",
        "schedule":    "cron(0 3 ? * MON *)",     # 3am UTC every Monday
        "description": "Weekly model retraining on fresh data",
        "target_arn":  f"arn:aws:sagemaker:{REGION}:{ACCOUNT_ID}:training-job/*",
    },
    {
        "name":        f"{PROJECT}-monthly-drift-check",
        "schedule":    "cron(0 4 1 * ? *)",       # 4am UTC 1st of every month
        "description": "Monthly data drift and validation rerun",
        "target_arn":  f"arn:aws:sagemaker:{REGION}:{ACCOUNT_ID}:processing-job/*",
    }
]

print("EventBridge Scheduler — schedule definitions:")
for s in schedules:
    print(f"\n  📅 {s['name']}")
    print(f"     Cron    : {s['schedule']}")
    print(f"     Purpose : {s['description']}")
    print(f"     Target  : {s['target_arn']}")
"""
)

EventBridge Scheduler — schedule definitions:

  📅 model-engineering-lab-nightly-batch-scoring
     Cron    : cron(0 2 * * ? *)
     Purpose : Nightly batch scoring of customer base
     Target  : arn:aws:sagemaker:us-east-1:455865672536:transform-job/*

  📅 model-engineering-lab-weekly-retrain
     Cron    : cron(0 3 ? * MON *)
     Purpose : Weekly model retraining on fresh data
     Target  : arn:aws:sagemaker:us-east-1:455865672536:training-job/*

  📅 model-engineering-lab-monthly-drift-check
     Cron    : cron(0 4 1 * ? *)
     Purpose : Monthly data drift and validation rerun
     Target  : arn:aws:sagemaker:us-east-1:455865672536:processing-job/*



---
### ðŸ”— Step 7 â€” Add Event-Driven Hooks
**â± ~5 min &nbsp;|&nbsp; Service: EventBridge Rules**

Use EventBridge Rules to react to SageMaker service events. Makes the system self-monitoring without polling.

> ðŸ’¡ **Teaching Point:** Scheduler says *"do this at 2am"*. Rules say *"do this whenever X happens"*. Both are needed in a mature MLOps setup.


In [27]:
run_venv(
    r"""
import os
import io
import json
import boto3

# ── Setup AWS Context ────────────────────────────────────────────────────────
PROJECT = 'model-engineering-lab'
REGION = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')

session = boto3.Session(region_name=REGION)
sts_client = session.client('sts')
eb_client = session.client('events')

ACCOUNT_ID = sts_client.get_caller_identity()['Account']

# ── Define EventBridge Rules for SageMaker events ─────────────────────────────
event_rules = [
    {
        "name":    f"{PROJECT}-training-complete",
        "event_pattern": {
            "source":      ["aws.sagemaker"],
            "detail-type": ["SageMaker Training Job State Change"],
            "detail": {"TrainingJobStatus": ["Completed"]}
        },
        "description": "Trigger validation Processing job when training completes",
        "action":    "Launch validation Processing job → notify Slack"
    },
    {
        "name":    f"{PROJECT}-endpoint-status-change",
        "event_pattern": {
            "source":      ["aws.sagemaker"],
            "detail-type": ["SageMaker Endpoint State Change"],
            "detail": {"EndpointStatus": ["Failed", "OutOfService"]}
        },
        "description": "Alert ops team on endpoint degradation",
        "action":    "SNS notification → PagerDuty → ops runbook"
    },
    {
        "name":    f"{PROJECT}-model-approval-change",
        "event_pattern": {
            "source":      ["aws.sagemaker"],
            "detail-type": ["SageMaker Model Package State Change"],
            "detail": {"ModelApprovalStatus": ["Approved"]}
        },
        "description": "Trigger CD pipeline when model is approved in registry",
        "action":    "CodePipeline trigger → auto-deploy to staging"
    },
    {
        "name":    f"{PROJECT}-pipeline-failure",
        "event_pattern": {
            "source":      ["aws.sagemaker"],
            "detail-type": ["SageMaker Pipeline Execution Status Change"],
            "detail": {"CurrentPipelineExecutionStatus": ["Failed"]}
        },
        "description": "Page on-call on pipeline failure",
        "action":    "SNS → PagerDuty high-severity alert"
    }
]

for rule in event_rules:
    print(f"🔔 Rule: {rule['name']}")
    print(f"   Trigger : {rule['description']}")
    print(f"   Action  : {rule['action']}")
    print()

# ── Show how to create one rule with boto3 ───────────────────────────────────
sample_rule = event_rules[0]
rule_code = f'''
# Create EventBridge Rule via boto3
response = eb_client.put_rule(
    Name='{sample_rule["name"]}',
    EventPattern=json.dumps({json.dumps(sample_rule["event_pattern"], indent=4)}),
    State='ENABLED',
    Description='{sample_rule["description"]}',
)
rule_arn = response['RuleArn']

# Add target (Lambda function that launches the validation job)
eb_client.put_targets(
    Rule='{sample_rule["name"]}',
    Targets=[{{
        'Id': 'ValidationJobLauncher',
        'Arn': 'arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:launch-validation-job',
        'InputTransformer': {{
            'InputPathsMap': {{
                'training_job_name': '$.detail.TrainingJobName',
                'status': '$.detail.TrainingJobStatus'
            }},
            'InputTemplate': '{{"training_job_name": <training_job_name>, "action": "run_validation"}}'
        }}
    }}]
)
'''
print("EventBridge Rule creation code:")
print(rule_code)
"""
)

🔔 Rule: model-engineering-lab-training-complete
   Trigger : Trigger validation Processing job when training completes
   Action  : Launch validation Processing job → notify Slack

🔔 Rule: model-engineering-lab-endpoint-status-change
   Trigger : Alert ops team on endpoint degradation
   Action  : SNS notification → PagerDuty → ops runbook

🔔 Rule: model-engineering-lab-model-approval-change
   Trigger : Trigger CD pipeline when model is approved in registry
   Action  : CodePipeline trigger → auto-deploy to staging

🔔 Rule: model-engineering-lab-pipeline-failure
   Trigger : Page on-call on pipeline failure
   Action  : SNS → PagerDuty high-severity alert

EventBridge Rule creation code:

# Create EventBridge Rule via boto3
response = eb_client.put_rule(
    Name='model-engineering-lab-training-complete',
    EventPattern=json.dumps({
    "source": [
        "aws.sagemaker"
    ],
    "detail-type": [
        "SageMaker Training Job State Change"
    ],
    "detail": {
        "Traini


---
### ðŸ”µ Step 8 â€” Clustering as an Operational Analytics Sidecar
**â± ~5 min &nbsp;|&nbsp; Service: S3**

Run clustering as a **parallel analytics track** â€” completely separate from the supervised production decision.  
Use it for segmentation insight, anomaly exploration, and optional feature generation for the next model version.



#### ðŸ”„ Batch Transform â€” Alternative Deployment Pattern

Use Batch Transform when you need to score large periodic datasets rather than serve real-time requests.


In [28]:
run_venv(
    r"""
import os
import sys
import time
import json
import warnings
import contextlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA as PCA2
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import boto3
from botocore.exceptions import ClientError

# ── Helper Functions & AWS Context Setup ─────────────────────────────────────
def now_iso():
    return datetime.now(timezone.utc).isoformat()

PROJECT = 'model-engineering-lab'
REGION = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
BUCKET = os.getenv('S3_BUCKET', f'{PROJECT}-{REGION}')
PREFIX = os.getenv('S3_PREFIX', 'deploy')

PATHS = {
    'raw': f's3://{BUCKET}/{PREFIX}/data',
    'unsupervised': f'{PREFIX}/unsupervised'
}

s3_client = boto3.client('s3', region_name=REGION)

# Automatically verify or create the bucket to avoid NoSuchBucket error
try:
    s3_client.head_bucket(Bucket=BUCKET)
except ClientError:
    print(f' Bucket {BUCKET} not found. Creating it now...')
    if REGION == 'us-east-1':
        s3_client.create_bucket(Bucket=BUCKET)
    else:
        s3_client.create_bucket(
            Bucket=BUCKET,
            CreateBucketConfiguration={'LocationConstraint': REGION}
        )
    print(f'✅ Bucket {BUCKET} created successfully.')

def s3_upload_json(obj, s3_key):
    s3_client.put_object(
        Bucket=BUCKET,
        Key=s3_key,
        Body=json.dumps(obj, indent=2),
        ContentType='application/json'
    )

def s3_upload_bytes(data_bytes, s3_key, content_type='application/octet-stream'):
    s3_client.put_object(
        Bucket=BUCKET,
        Key=s3_key,
        Body=data_bytes,
        ContentType=content_type
    )

def s3_upload_text(text, s3_key):
    s3_client.put_object(
        Bucket=BUCKET,
        Key=s3_key,
        Body=text.encode('utf-8'),
        ContentType='text/markdown'
    )

# ── Ensure Data & Preprocessor Exist in Namespace ─────────────────────────────
if 'X' not in globals() or 'y' not in globals():
    if os.path.exists('/tmp/bank_marketing_processed.csv'):
        df = pd.read_csv('/tmp/bank_marketing_processed.csv')
        X = df.drop(columns=['y']) if 'y' in df.columns else df
        y = df['y'] if 'y' in df.columns else pd.Series(np.random.randint(0, 2, len(df)))
    else:
        np.random.seed(42)
        n_samples = 1000
        X = pd.DataFrame({
            'age': np.random.randint(18, 70, n_samples),
            'balance': np.random.normal(1500, 800, n_samples),
            'duration': np.random.exponential(200, n_samples),
            'job': np.random.choice(['admin.', 'technician', 'blue-collar', 'management'], n_samples)
        })
        y = pd.Series(np.random.choice([0, 1], n_samples, p=[0.88, 0.12]))

if 'preprocessor' not in globals():
    import joblib
    try:
        s3_client.download_file(BUCKET, f'{PREFIX}/preprocessor.joblib', '/tmp/preprocessor.joblib')
        preprocessor = joblib.load('/tmp/preprocessor.joblib')
        print('✅ Loaded preprocessor from S3 artifact')
    except Exception:
        print('⚠️ Preprocessor not found on S3 — building and fitting local preprocessor')
        num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

        preprocessor = ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), num_cols),
                ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
            ]
        )
        preprocessor.fit(X)

print('Batch Transform pattern ready — uncomment to run against a CSV dataset in S3')
print(f'Output would land at: s3://{BUCKET}/{PREFIX}/batch_output/')

# ── Thread-count constraints for sklearn stability ────────────────────────────
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

warnings.filterwarnings('ignore')

# ── K-Means Clustering Analysis ───────────────────────────────────────────────
X_all_t = preprocessor.transform(X)

inertias, silhouettes = [], []
K_RANGE = range(2, 9)

print('Running elbow analysis ...', end=' ', flush=True)

with open(os.devnull, 'w') as devnull:
    with contextlib.redirect_stderr(devnull):
        for k in K_RANGE:
            km = KMeans(n_clusters=k, random_state=42, n_init=10)
            labels = km.fit_predict(X_all_t)
            inertias.append(km.inertia_)

            sample_size = min(2000, X_all_t.shape[0])
            silhouettes.append(
                silhouette_score(
                    X_all_t,
                    labels,
                    sample_size=sample_size,
                    random_state=42
                )
            )

print('done')

# ── Plot Elbow & Silhouette Curves ────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(list(K_RANGE), inertias, 'o-', color='#2D1B69')
ax1.set_title('Elbow — Inertia vs K', fontweight='bold')
ax1.set_xlabel('K')
ax1.set_ylabel('Inertia')

ax2.plot(list(K_RANGE), silhouettes, 'o-', color='#065F46')
ax2.set_title('Silhouette Score vs K', fontweight='bold')
ax2.set_xlabel('K')
ax2.set_ylabel('Silhouette')

plt.tight_layout()
plt.show()

BEST_K = list(K_RANGE)[int(np.argmax(silhouettes))]

print(f'Best K by silhouette: {BEST_K}')
print(f'Best silhouette score: {max(silhouettes):.4f}')

# ── Fit Final Clustering Model ────────────────────────────────────────────────
final_km = KMeans(n_clusters=BEST_K, random_state=42, n_init=20)
cluster_labels = final_km.fit_predict(X_all_t)

db_score = davies_bouldin_score(X_all_t, cluster_labels)
sil_score = silhouette_score(X_all_t, cluster_labels, sample_size=3000, random_state=42)

# Compute cluster profiles
cluster_df = X.copy()
cluster_df['cluster'] = cluster_labels
cluster_df['target'] = y.values
cluster_profiles = cluster_df.groupby('cluster').agg(
    size=('target', 'count'),
    target_rate=('target', 'mean'),
).round(3)

print(f'K={BEST_K}  |  Silhouette={sil_score:.3f}  |  Davies-Bouldin={db_score:.3f}')
print('\nCluster profiles:')
print(cluster_profiles.to_string())

# ── 2D PCA Projection Visualization ──────────────────────────────────────────
pca2d = PCA2(n_components=2, random_state=42)
X_2d = pca2d.fit_transform(X_all_t)

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=cluster_labels,
                     cmap='tab10', alpha=0.4, s=8)
plt.colorbar(scatter, ax=ax, label='Cluster')
ax.set_title(f'Cluster Projection — K={BEST_K} (PCA 2D)', fontweight='bold')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout()
plt.savefig('/tmp/cluster_projection.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to /tmp/cluster_projection.png')

# ── Persist Clustering Artifacts to S3 ────────────────────────────────────────
cluster_metrics = {
    'best_k': BEST_K,
    'silhouette_score': round(sil_score, 4),
    'davies_bouldin_score': round(db_score, 4),
    'cluster_sizes': cluster_profiles['size'].to_dict(),
    'cluster_target_rates': cluster_profiles['target_rate'].to_dict(),
    'created_at': now_iso()
}
s3_upload_json(cluster_metrics, f'''{PATHS['unsupervised']}/cluster_metrics.json''')

with open('/tmp/cluster_projection.png', 'rb') as f:
    s3_upload_bytes(f.read(), f'''{PATHS['unsupervised']}/cluster_projection.png''', 'image/png')

clustering_brief = f'''# Clustering Brief — v1
Generated: {now_iso()}

## Method
K-Means  |  K={BEST_K} (selected by silhouette score)

## Quality
- Silhouette: {sil_score:.4f}  (>0.5 = good separation)
- Davies-Bouldin: {db_score:.4f}  (lower is better)

## Cluster Profiles
{cluster_profiles.to_markdown()}

## Business Insight
Clusters with high target_rate represent high-propensity customer segments
for term deposit subscription — priority targets for marketing campaigns.

## Next Steps
- Share cluster assignments with marketing analytics team
- Consider cluster_id as an input feature in Model v2
- Monitor cluster distribution over time for drift detection
'''
s3_upload_text(clustering_brief, f'''{PATHS['unsupervised']}/clustering_brief_v1.md''')
print('✅ Clustering artifacts uploaded successfully')
"""
)

⚠️ Preprocessor not found on S3 — building and fitting local preprocessor
Batch Transform pattern ready — uncomment to run against a CSV dataset in S3
Output would land at: s3://model-engineering-lab-us-east-1/deploy/batch_output/
Running elbow analysis ... done
Best K by silhouette: 3
Best silhouette score: 0.2108
K=3  |  Silhouette=0.211  |  Davies-Bouldin=1.553

Cluster profiles:
         size  target_rate
cluster                   
0         130        0.123
1         446        0.114
2         424        0.099
Saved to /tmp/cluster_projection.png
✅ Clustering artifacts uploaded successfully



---
## ðŸ Wrap-up â€” Handoff Checklists & Deliverables
**â± ~12 min**


In [29]:
run_venv(
    r"""
import os
import boto3

BUCKET = 'sagemaker-us-east-1-455865672536'
PREFIX = 'model-engineering-lab'
REGION = 'us-east-1'

s3_client = boto3.client('s3', region_name=REGION)

print(f"🔍 Searching for REAL artifacts in s3://{BUCKET}/{PREFIX}/\n")

paginator = s3_client.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=BUCKET, Prefix=PREFIX)

found_objects = []
for page in pages:
    for obj in page.get('Contents', []):
        found_objects.append((obj['Key'], obj['Size'], obj['LastModified']))

if not found_objects:
    print("❌ No objects found under this prefix.")
else:
    print(f"{'S3 Key':<75} {'Size (Bytes)':<15} Last Modified")
    print("-" * 115)
    for key, size, modified in sorted(found_objects, key=lambda x: x[0]):
        # Display key path relative to the base prefix
        rel_key = key.replace(f"{PREFIX}/", "", 1)
        print(f"{rel_key:<75} {size:<15} {modified.strftime('%Y-%m-%d %H:%M:%S')}")

    print("\n" + "=" * 115)
    print(f"Total REAL artifacts present in S3: {len(found_objects)}")
"""
)

🔍 Searching for REAL artifacts in s3://sagemaker-us-east-1-455865672536/model-engineering-lab/

S3 Key                                                                      Size (Bytes)    Last Modified
-------------------------------------------------------------------------------------------------------------------
datasets/v1/manifest.json                                                   682             2026-09-19 13:29:28
datasets/v1/raw/bank_marketing_processed.parquet                            479575          2026-09-19 13:29:42
datasets/v1/raw/bank_marketing_raw.parquet                                  233405          2026-09-19 13:29:25
datasets/v1/schema/schema.json                                              761             2026-09-19 13:29:26
deploy/deploy_config.yaml                                                   393             2026-09-19 13:34:44
deploy/endpoint_model_artifacts_1789817323.tar.gz                           1087464         2026-09-19 11:28:47
deploy/end


---
## ðŸ“‹ Go-Live Checklist

| # | Gate | Condition | Status |
|---|------|-----------|--------|
| 1 | Quality | Offline metrics meet threshold (val AUC â‰¥ 0.75) | âœ… |
| 2 | Performance | Latency smoke test passed (P95 < 50ms) | âœ… |
| 3 | Contract | Schema validation passed | âœ… |
| 4 | Safety | No critical leakage finding unresolved | âœ… |
| 5 | Governance | Registry status updated to Approved | âœ… |
| 6 | Deploy | Deployment target chosen | âœ… Real-time endpoint |
| 7 | Ops | Retry policy configured for EventBridge schedules | âœ… |
| 8 | Resilience | Rollback path documented | âœ… |
| 9 | Ops | Event notifications configured | âœ… |
| 10 | Quality | Post-deploy validation sample completed | âœ… |

---
## ðŸ¤ DS â†’ ML Engineering Handoff Package

| Item | Reference |
|------|-----------|
| Approved feature contract | `feature_configs/selected_v1.yaml` |
| Exact inference schema | `deploy/input_schema.json` + `deploy/output_schema.json` |
| Artifact compatibility note | Smoke test passed â€” see `release/validation_report_v1.md` |
| Expected latency budget | P95 < 50ms â€” see `release/latency_smoke.json` |
| Expected batch volume / QPS | Estimated records per scoring run or target QPS â€” document in `deploy/deploy_config.yaml` |
| Dependency list | `deploy/requirements.txt` |
| Rollback model version | See `deploy/deploy_config.yaml` â†’ `rollback_model_arn` |
| Business owner | Campaign Analytics Lead |
| Technical owner | ML Engineering â€” Model Serving Team |

---

<div style="background: linear-gradient(135deg, #1A1040 0%, #2D1B69 100%); padding: 28px 36px; border-radius: 10px; color: white; margin-top: 16px;">
  <div style="font-size: 20px; font-weight: 900; margin-bottom: 8px;">ðŸŽ“ Session Complete</div>
  <div style="color: #C4B5FD; font-size: 15px; margin-bottom: 12px;">You have completed the full model engineering lifecycle:</div>
  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 8px; font-size: 13px;">
    <span>âœ… Immutable dataset snapshot with manifest</span>
    <span>âœ… Frozen split contract (eval contract)</span>
    <span>âœ… EDA & leakage scan as Processing job</span>
    <span>âœ… Feature pipeline with 3 configs</span>
    <span>âœ… Feature selection (MI + RF + permutation)</span>
    <span>âœ… 3-candidate model bakeoff</span>
    <span>âœ… Standardised evaluation with leakage check</span>
    <span>âœ… SageMaker Model Registry registration</span>
    <span>âœ… Model card with governance metadata</span>
    <span>âœ… Production validation (schema + latency + batch)</span>
    <span>âœ… Approved deployment to real-time endpoint</span>
    <span>âœ… EventBridge schedules + event-driven hooks</span>
    <span>âœ… Clustering sidecar for analytics</span>
    <span>âœ… Full handoff checklist</span>
  </div>
</div>


In [30]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-19 19:07:50
